<a href="https://colab.research.google.com/github/yangyi02/droid/blob/main/Copy_of_2026_05_18_DROID_dataset_v52.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### DROID: A Large-Scale In-The-Wild Robot Manipulation Dataset

![](https://droid-dataset.github.io/droid/assets/index/droid_teaser.jpg)

This Colab demonstrates how to load and visualize samples from the DROID dataset. Please also check out our [dataset visualizer](https://droid-dataset.github.io/dataset.html) to explore the dataset.

You can download the full dataset (1.7TB) using:
```
gsutil -m cp -r gs://gresearch/robotics/droid <your_local_path>
```

If you'd like to download an example version of the dataset with 100 episodes first (2GB), run:
```
gsutil -m cp -r gs://gresearch/robotics/droid_100 <your_local_path>
```

If you want to use DROID for policy training, please check out our [policy training repo](https://github.com/droid-dataset/droid_policy_learning).

### 环境配置与初始化

In [ ]:
# @title 安装 Python 依赖库

!pip install mediapy pyrender trimesh PyOpenGL-accelerate open3d pybullet polyscope yourdfpy pytorch_kinematics

In [ ]:
# @title 导入通用 Python 库

import copy
import os
import sys
import gc
import glob
import cv2
import h5py
import json
from matplotlib import cm
import matplotlib.pyplot as plt
import mediapy as media
import numpy as np
from PIL import Image, ImageDraw
import plotly.graph_objects as go
import plotly.io as pio
import polyscope as ps
import pybullet as p
import pybullet_data
import pyrender
import random
from scipy.spatial.transform import Rotation as R, Slerp
import trimesh
import tensorflow_datasets as tfds
from tqdm import tqdm
import torch
import torch.nn.functional as F
import torch.optim as optim

os.environ['PYOPENGL_PLATFORM'] = 'egl'
pio.renderers.default = 'colab'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

In [ ]:
# @title 克隆 Github 仓库 (VGGT, CoTracker)

%cd /content

!git clone https://github.com/facebookresearch/vggt.git
sys.path.append("/content/vggt")

!git clone https://github.com/facebookresearch/co-tracker.git
sys.path.append("/content/co-tracker")

from vggt.models.vggt import VGGT
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
print("✅ [2/4] VGGT 视觉外参大模型模块 导入成功！")

from cotracker.predictor import CoTrackerPredictor
print("✅ [3/4] CoTracker3 稠密点追踪模块 导入成功！")

In [ ]:
# @title 下载与初始化 CoTracker3 模型

from cotracker.predictor import CoTrackerPredictor

WEIGHTS_URL = "https://huggingface.co/facebook/cotracker3/resolve/main/scaled_offline.pth"
os.makedirs("/content/co-tracker/weights", exist_ok=True)
weights_path = os.path.join("/content/co-tracker/weights/", "cotracker3_offline.pth")

if not os.path.exists(weights_path):
    !wget -q {WEIGHTS_URL} -O {weights_path}

cotracker_model = CoTrackerPredictor(checkpoint=weights_path)
cotracker_model = cotracker_model.to(device)

print("✅ 模型加载成功！现在可以继续运行后续的 2D 跟踪代码了。")

In [ ]:
# @title 下载与初始化 VGGT 模型

from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri

print("🚀 正在加载 VGGT-1B 模型至显存...")
vggt_model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)
vggt_model.eval()
print("✅ VGGT 模型加载完成！")

# 1. 模型懒加载机制 (保护显存)
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

def estimate_multi_camera_vggt(img_list):
    """
    V14 核心升华版：支持任意数量 N 个相机的全局联合推理！
    第 0 张图被视为参考系原点，返回后续所有图相对于原点的 T 矩阵。
    """
    filenames = []
    # 动态保存临时文件以适配 VGGT 的预处理接口
    for i, img in enumerate(img_list):
        fname = f"tmp_vggt_{i}.png"
        cv2.imwrite(fname, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
        filenames.append(fname)

    images = load_and_preprocess_images(filenames).to(device)
    images_input = images.unsqueeze(0) # [1, N, 3, H, W]

    with torch.inference_mode():
        with torch.autocast(device_type="cuda", dtype=dtype):
            aggregated_tokens_list, ps_idx = vggt_model.aggregator(images_input)
            pose_enc = vggt_model.camera_head(aggregated_tokens_list)[-1]
            extrinsic, intrinsic = pose_encoding_to_extri_intri(pose_enc, images_input.shape[-2:])

    # 提取第 0 张图 (参考图) 到其他所有图的变换矩阵 T_{ref -> tgt}
    T_ref_to_tgts = []
    for i in range(1, len(img_list)):
        ext_mat = extrinsic[0, i].cpu().numpy()
        T = np.eye(4)
        T[:3, :] = ext_mat
        T_ref_to_tgts.append(T)

    return T_ref_to_tgts

In [ ]:
# @title 下载 Robotiq URDF 与 3D 物理资产

# 1. 克隆包含定制版 Franka + Robotiq 模型的官方仓库 (强行指定 data 分支)
!git clone -b data https://github.com/NVlabs/PointWorld.git

In [ ]:
# @title 下载并解析 DROID 数据集元数据 (JSON)

root_path = "/content/droid_raw/1.0.1"
base_url = "https://huggingface.co/KarlP/droid/resolve/main"

# 🗑️ 移除 cam2base_extrinsic_superset.json
files = ["intrinsics.json", "camera_serials.json", "episode_id_to_path.json", "keep_ranges_1_0_1.json", "cam2base_extrinsic_superset.json"]

os.makedirs(root_path, exist_ok=True)
for f in files:
    os.system(f"wget -q -nc -P {root_path} {base_url}/{f}")

def load_json(name):
    with open(os.path.join(root_path, name), 'r') as f: return json.load(f)

serials_db, id_to_path = load_json(files[1]), load_json(files[2])
keep_ranges = load_json(files[3])
extrinsics_db = load_json(files[4])

# 🌟 解锁封印：现在的 valid_ids 只受限于基本元数据，直接打通全部数据！
valid_ids = sorted(set(serials_db.keys()) & set(id_to_path.keys()))
print(f"✅ 全局元数据准备完毕！匹配到 {len(valid_ids)} 个 Episode。")

# 🌟 新增：计算包含官方外参的 Episode 集合
episodes_with_ext = set(valid_ids) & set(extrinsics_db.keys())
print(f"   📸 其中包含官方预标定外参的 Episode 数量为: {len(episodes_with_ext)} 个。")

# 只采样有初始相机标定的数据
valid_ids = sorted(set(serials_db.keys()) & set(id_to_path.keys()) & set(extrinsics_db.keys()))
print(f"✅ 全局元数据准备完毕！匹配到包含官方预标定外参的 Episode 共 {len(valid_ids)} 个。")

In [ ]:
# @title 已处理的 episode
valid_ids = [
    "PennPAL+c5f808b7+2023-06-15-17h-12m-39s",
    "REAL+de601749+2023-12-19-18h-19m-27s",
    "REAL+de601749+2023-12-19-18h-19m-27s",
    "GuptaLab+553d1bd5+2023-04-30-16h-07m-27s",
    "IPRL+edf28ef3+2024-01-01-10h-36m-18s",
    "GuptaLab+553d1bd5+2023-05-28-16h-48m-13s",
    "PennPAL+c5f808b7+2023-06-15-17h-12m-39s",
    "TRI+52ca9b6a+2023-12-05-15h-19m-45s",
]

### 数据准备

In [ ]:
# @title 下载 DROID Episode 并初始化全局状态池

def download_episode(episode_id, root_path, id_to_path, serials_db, keep_ranges_db):
    """一站式下载视频，并按层级分类初始化 scene_constants (元数据终极聚合版)"""
    # id_to_path[episode_id] 本身就是类似 "RPL/failure/2023-11-30-18h-16m-00s" 的相对路径
    relative_path = id_to_path[episode_id]
    episode_path = os.path.join(root_path, relative_path)

    if not os.path.exists(episode_path):
        os.makedirs(episode_path, exist_ok=True)
        os.system(f"gsutil -m -q cp -r \"gs://gresearch/robotics/droid_raw/1.0.1/{relative_path}/*\" \"{episode_path}/\"")
        print(f"  ⬇️ 成功下载数据: {episode_id}")
    else:
        print(f"  ⏭️ 数据已存在，跳过下载。")

    cam_info = serials_db[episode_id]
    wrist_serial = cam_info.get('wrist_cam_serial')

    # 🗑️ 彻底脱离 intrinsics_db 依赖，直接从串口数据中提取全部机位
    valid_cams = sorted(set(cam_info.values()))

    # 🌟 修复：完美还原官方 JSON 里的 absolute bucket path 格式
    base_prefix = "gs://xembodiment_data/r2d2/r2d2-data-full/"
    episode_key = f"{base_prefix}{relative_path}/recordings/MP4--{base_prefix}{relative_path}/trajectory.h5"

    valid_indices = None

    if episode_key in keep_ranges_db:
        ranges = keep_ranges_db[episode_key]
        indices = []
        for start, end in ranges:
            indices.extend(range(start, end))
        valid_indices = np.array(indices)
        print(f"  ✂️ 已预加载动作区间，共标记 {len(valid_indices)} 帧为有效关键帧。")
    else:
        # 为了防范极端情况，如果真找不到（尽管概率很小），保留全量帧
        print(f"  ⚠️ 未找到 {episode_id} 的 Idle 过滤信息，将默认保留全量帧。")

    # 🌟 极简封箱：结构化分为 meta, robot, camera 三大核心模块
    return {
        'meta': {
            'episode_id': episode_id,
            'episode_path': episode_path,
            'wrist_serial': wrist_serial,
            'valid_indices': valid_indices  # <--- 完美注入！
        },
        'robot': {},  # 预留位，供后续运动学解析填入
        'camera': {
            cam: {
                'baseline': 0.063 if cam == wrist_serial else 0.120,
            } for cam in valid_cams  # 字典推导式优雅注入所有相机
        }
    }

# ================= 主流程 =================
episode_id = random.choice(valid_ids)
print(f"🎯 选定处理的 Episode: {episode_id}")

# 🌟 移除了 intrinsics_db 传参
scene_constants = download_episode(episode_id, root_path, id_to_path, serials_db, keep_ranges)

print(f"✅ scene_constants 初始化就绪！")
print(f"   - 腕部相机: {scene_constants['meta']['wrist_serial']}")
print(f"   - 已加载机位: {list(scene_constants['camera'].keys())}")
if scene_constants['meta']['valid_indices'] is not None:
    print(f"   - 动作有效帧数: {len(scene_constants['meta']['valid_indices'])}")

In [ ]:
# @title 从 Stage1 Bucket 加载确切的文件 (免登录匿名版 + 严格模式)

def load_stage1_camera_data(scene_constants, bucket_prefix="gs://dm-tapnet/mv-tap/droid/depth"):
    episode_id = scene_constants['meta']['episode_id']
    local_cache_dir = f"/content/droid_depth_cache/{episode_id}"
    os.makedirs(local_cache_dir, exist_ok=True)

    print(f"  ☁️ 免授权极速加载: 绕过目录遍历，直接拉取 {episode_id} 的确切文件...")

    # ==========================================
    # 🌟 1. 匿名下载并解析 Robot 数据
    # ==========================================
    robot_gcs_path = f"{bucket_prefix}/{episode_id}/robot.npz"
    robot_local_path = os.path.join(local_cache_dir, "robot.npz")

    # 显式下载确切的单文件，绝不使用通配符 (*)
    os.system(f"gsutil cp '{robot_gcs_path}' '{local_cache_dir}/' > /dev/null 2>&1")

    # 强制读取 (如果文件不存在直接报错 FileNotFoundError)
    robot_data = np.load(robot_local_path, allow_pickle=True)

    # 安全读取 robot.npz 中的变量
    for k in ['joint_positions', 'gripper_positions', 'T_cam_ee_init', 'T_ee_base_all']:
        if k in robot_data:
            scene_constants['robot'][k] = robot_data[k]

    if 'valid_indices' in robot_data:
        scene_constants['meta']['valid_indices'] = robot_data['valid_indices']
    if 'wrist_serial' in robot_data:
        scene_constants['meta']['wrist_serial'] = str(robot_data['wrist_serial'].item())

    wrist_serial = scene_constants['meta'].get('wrist_serial')
    print(f"    ✅ 成功匿名拉取并挂载 robot.npz")

    # ==========================================
    # 🌟 2. 匿名并发下载并解析各相机数据
    # ==========================================
    # 所有相机共有的基础文件
    base_cam_files = [
        "video_left.mp4", "video_right.mp4",
        "video_left_raw.mp4", "video_right_raw.mp4",
        "raw_depth.npz", "calibration.npz"
    ]

    for cam in scene_constants['camera']:
        local_cam_dir = os.path.join(local_cache_dir, cam)
        os.makedirs(local_cam_dir, exist_ok=True)

        # 如果是腕部相机，追加专属的夹爪提纯工件
        cam_files = list(base_cam_files)
        if cam == wrist_serial:
            cam_files.extend([
                "original_raw_depth.npz",
                "gripper_mask.npz",
                "gripper_depth.npz"
            ])

        # 拼接出极其精确的 GCS 路径
        gcs_files = [f"'{bucket_prefix}/{episode_id}/{cam}/{fname}'" for fname in cam_files]
        gcs_files_str = " ".join(gcs_files)

        # 核心黑科技：一股脑丢给 gsutil，它会自动并发 Get，完美绕过 List 校验
        os.system(f"gsutil -m cp {gcs_files_str} '{local_cam_dir}/' > /dev/null 2>&1")

        # --- 强制读取 4 路 MP4 视频 ---
        video_keys = {
            "video_rgb": "video_left.mp4",
            "video_right": "video_right.mp4",
            "video_raw_rgb": "video_left_raw.mp4",
            "video_raw_right": "video_right_raw.mp4",
        }
        for mem_key, filename in video_keys.items():
            vid_path = os.path.join(local_cam_dir, filename)
            if os.path.exists(vid_path):
                scene_constants['camera'][cam][mem_key] = media.read_video(vid_path)

        # --- 读取深度图并还原单位 (mm -> meters) ---
        depth_path = os.path.join(local_cam_dir, "raw_depth.npz")
        if os.path.exists(depth_path):
            scene_constants['camera'][cam]['raw_depth'] = np.load(depth_path)['depth'].astype(np.float32) / 1000.0

        # --- 读取腕部专属工件 ---
        orig_depth_path = os.path.join(local_cam_dir, "original_raw_depth.npz")
        if os.path.exists(orig_depth_path):
            scene_constants['camera'][cam]['original_raw_depth'] = np.load(orig_depth_path)['depth'].astype(np.float32) / 1000.0

        mask_path = os.path.join(local_cam_dir, "gripper_mask.npz")
        if os.path.exists(mask_path):
            scene_constants['camera'][cam]['sam_real_masks'] = np.load(mask_path)['mask']

        grip_depth_path = os.path.join(local_cam_dir, "gripper_depth.npz")
        if os.path.exists(grip_depth_path):
            scene_constants['camera'][cam]['empirical_gripper_depth'] = np.load(grip_depth_path)['depth'].astype(np.float32) / 1000.0

        # --- 强制重构标定参数字典 ---
        calib_path = os.path.join(local_cam_dir, "calibration.npz")
        if os.path.exists(calib_path):
            calib_npz = np.load(calib_path)
            scene_constants['camera'][cam]['K_mat'] = calib_npz['K_calib_left']
            scene_constants['camera'][cam]['baseline'] = float(calib_npz['baseline'])

            scene_constants['camera'][cam]['zed_calibration'] = {
                'calibrated': {
                    'K': calib_npz['K_calib_left'], 'disto': calib_npz['disto_calib_left'],
                    'K_right': calib_npz['K_calib_right'], 'disto_right': calib_npz['disto_calib_right']
                },
                'raw': {
                    'K': calib_npz['K_raw_left'], 'disto': calib_npz['disto_raw_left'],
                    'K_right': calib_npz['K_raw_right'], 'disto_right': calib_npz['disto_raw_right']
                }
            }

        print(f"    ✅ 成功匿名拉取相机 {cam} 的全部数据")

    return scene_constants

# ================= 主流程 =================
scene_constants = load_stage1_camera_data(scene_constants)

In [ ]:
# @title 打印当前读取内容

def inspect_dict_structure(data, name="scene_constants", indent=0):
    """递归打印字典内各变量的形状、长度和类型"""
    spacing = "  " * indent
    if isinstance(data, dict):
        print(f"{spacing}📂 {name} (dict, {len(data)} keys)")
        for k, v in data.items():
            inspect_dict_structure(v, name=str(k), indent=indent + 1)
    elif isinstance(data, np.ndarray):
        print(f"{spacing}📊 {name}: ndarray, shape={data.shape}, dtype={data.dtype}")
    elif torch.is_tensor(data):
        print(f"{spacing}🔥 {name}: Tensor, shape={tuple(data.shape)}, dtype={data.dtype}")
    elif isinstance(data, (list, tuple)):
        print(f"{spacing}📜 {name}: {type(data).__name__}, len={len(data)}")
    else:
        # 对于字符串、浮点数等基础类型，直接打印前 50 个字符
        val_str = str(data)
        if len(val_str) > 50:
            val_str = val_str[:47] + "..."
        print(f"{spacing}🏷️ {name}: {type(data).__name__} = {val_str}")

# 执行全域透视
print("\n" + "="*50)
inspect_dict_structure(scene_constants)
print("="*50 + "\n")

In [ ]:
# @title 视差视频可视化 (跳过 S2M2，直接渲染已有深度)

def visualize_disparity_video(disp_array, vmax=100.0):
    """向量化预处理 + 优雅推导式完成视差图染色"""
    disp_norm = (np.clip(disp_array, 0, vmax) / vmax * 255).astype(np.uint8)

    return np.stack([
        cv2.cvtColor(cv2.applyColorMap(frame, cv2.COLORMAP_MAGMA), cv2.COLOR_BGR2RGB)
        for frame in disp_norm
    ])

def render_multicam_disparity_video(scene_constants, tgt_size=(128, 228), disp_vmax=100.0):
    """一站式提取常量池数据，生成 [左 | 右 | 视差] 的多视角拼接视频"""
    camera_rows = []

    # 纯粹的遍历：直接迭代 camera 层级
    for cam_data in scene_constants['camera'].values():

        # 1. 语义复苏：提取并缩放左右眼视频画面
        left_video = media.resize_video(cam_data['video_rgb'], tgt_size)
        right_video = media.resize_video(cam_data['video_right'], tgt_size)

        # 🌟 2. 视差还原：从存储的 depth 中反向推导 disparity 用于可视化渲染
        raw_depth = cam_data['raw_depth'].astype(np.float32)
        fx = cam_data['K_mat'][0, 0]
        baseline = cam_data['baseline']

        # 反向物理公式: d = (fx * baseline) / Z
        raw_disp = np.zeros_like(raw_depth)
        valid_mask = raw_depth > 0
        raw_disp[valid_mask] = (fx * baseline) / raw_depth[valid_mask]

        # 缩放 -> 渲染染色
        disp_video = visualize_disparity_video(media.resize_video(raw_disp, tgt_size), vmax=disp_vmax)

        # 3. 画面拼接：横向连接单个机位的三重视角 (Width 维度)
        camera_rows.append(np.concatenate([left_video, right_video, disp_video], axis=2))

    # 终极组装：纵向堆叠所有机位的画面 (Height 维度)
    return np.concatenate(camera_rows, axis=1)

# ================= 主流程 =================
# 🚀 移除 scene_constants = compute_stereo_depth(...)
# 直接利用已经存在 scene_constants 里的 raw_depth 生成预览视频！
final_video = render_multicam_disparity_video(scene_constants)
media.show_video(final_video, codec='gif')

In [ ]:
# @title 🎯 3D 视觉几何核心大一统算子库

def decode_disparity_np(disp, fx, baseline):
    """将原始视差转化为物理深度 (NumPy)"""
    z = np.zeros_like(disp)
    valid_mask = disp > 0  # 过滤无效视差
    z[valid_mask] = (fx * baseline) / disp[valid_mask]
    return z

def decode_disparity_pt(disp, fx, baseline):
    """将原始视差转化为物理深度 (PyTorch 完美梯度版)"""
    z = torch.zeros_like(disp)
    valid_mask = disp > 0  # 过滤无效视差
    z[valid_mask] = (fx * baseline) / disp[valid_mask]
    return z

def unproject_points_np(u, v, z, K, T_cam2world=None):
    """纯粹的 3D 射线反投影算子，只处理合法的物理深度 Z (NumPy)"""
    x_cam = (u - K[0, 2]) * z / K[0, 0]
    y_cam = (v - K[1, 2]) * z / K[1, 1]

    pts_cam = np.stack([x_cam, y_cam, z, np.ones_like(z)], axis=0)

    if T_cam2world is None:
        return pts_cam[:3, :].T
    return (T_cam2world @ pts_cam)[:3, :].T

def unproject_points_pt(u, v, z, K, T_cam2world=None):
    """纯粹的 3D 射线反投影算子，完美保持梯度穿透 (PyTorch)"""
    x_cam = (u - K[0, 2]) * z / K[0, 0]
    y_cam = (v - K[1, 2]) * z / K[1, 1]

    pts_cam = torch.stack([x_cam, y_cam, z, torch.ones_like(z)], dim=0)

    if T_cam2world is None:
        return pts_cam[:3, :].T
    return (T_cam2world @ pts_cam)[:3, :].T

def project_points_np(pts_world, K, T_cam2world):
    """将 3D 世界点阵拍回 2D 像素平面 (NumPy)"""
    T_world2cam = np.linalg.inv(T_cam2world)
    pts_homo = np.hstack([pts_world, np.ones((len(pts_world), 1))]).T
    pts_cam = T_world2cam @ pts_homo

    z_cam = pts_cam[2, :]
    u = np.zeros_like(pts_cam[0, :])
    v = np.zeros_like(pts_cam[1, :])

    valid_mask = z_cam > 0
    u[valid_mask] = (pts_cam[0, valid_mask] / z_cam[valid_mask]) * K[0, 0] + K[0, 2]
    v[valid_mask] = (pts_cam[1, valid_mask] / z_cam[valid_mask]) * K[1, 1] + K[1, 2]

    return u, v, z_cam

def project_points_pt(pts_world, K, T_cam2world):
    """将 3D 世界点阵拍回 2D 像素平面，完美保留计算图 (PyTorch)"""
    T_world2cam = torch.linalg.inv(T_cam2world)
    pts_homo = torch.cat([pts_world, torch.ones((len(pts_world), 1), device=pts_world.device)], dim=1).T
    pts_cam = T_world2cam @ pts_homo

    z_cam = pts_cam[2, :]
    u = torch.zeros_like(pts_cam[0, :])
    v = torch.zeros_like(pts_cam[1, :])

    valid_mask = z_cam > 0
    u[valid_mask] = (pts_cam[0, valid_mask] / z_cam[valid_mask]) * K[0, 0] + K[0, 2]
    v[valid_mask] = (pts_cam[1, valid_mask] / z_cam[valid_mask]) * K[1, 1] + K[1, 2]

    return u, v, z_cam

In [ ]:
# @title 点云可视化函数

def unproject_to_3d(depth, color_img, K_mat, T_cam2world=None, min_depth=0., max_depth=1.5):
    """纯粹的几何升维算子：输入已校准的深度，专心做空间截断与反投影"""
    # 🌟 1. 空间屏蔽：用最符合直觉的物理阈值过滤
    mask = (depth > min_depth) & (depth < max_depth)
    v, u = np.where(mask)

    # 🌟 2. 几何升维：调用底层原生算子
    if T_cam2world is None:
        T_cam2world = np.eye(4)
    pts_world = unproject_points_np(u, v, depth[mask], K_mat, T_cam2world)

    return pts_world, color_img[mask]

def show_plotly_point_cloud(pts, cols, title="3D Point Cloud", max_points=150000, eye_pos=(-1.5, -1.5, 1.0)):
    """交互式点云渲染 (保持极简瘦身版不变)"""
    idx = np.random.permutation(len(pts))[:max_points]
    p, c = pts[idx], cols[idx]

    go.Figure(
        data=[go.Scatter3d(x=p[:, 0], y=p[:, 1], z=p[:, 2], mode='markers',
                           marker=dict(size=1.5, color=[f'rgb({r},{g},{b})' for r, g, b in c]))],
        layout=go.Layout(
            title=title, margin=dict(l=0, r=0, b=0, t=40), height=500, showlegend=False,
            scene=dict(aspectmode='data', camera=dict(eye=dict(x=eye_pos[0], y=eye_pos[1], z=eye_pos[2])))
        )
    ).show(renderer="colab")

In [ ]:
# @title 🎯 展示预加载的极品静态遮罩 (跳过 SAM 推断版)

def visualize_preloaded_gripper_mask(scene_constants):
    wrist_serial = scene_constants['meta']['wrist_serial']
    cam_data = scene_constants['camera'][wrist_serial]

    if 'sam_real_masks' not in cam_data:
        print("⚠️ 未找到预加载的 sam_real_masks，请确保数据加载步骤成功读取了 gripper_mask.npz！")
        return scene_constants

    gripper_states = scene_constants['robot']['gripper_positions']
    closed_indices = np.where(gripper_states < 0.05)[0]

    if len(closed_indices) == 0:
        print("⚠️ 该片段无夹爪完全闭合帧，采用第 0 帧作为底图展示。")
        ref_idx = 0
    else:
        ref_idx = closed_indices[0]

    # 取出底图和对应的静态遮罩
    ref_img = cam_data['video_rgb'][ref_idx].copy()
    final_mask = cam_data['sam_real_masks'][ref_idx]

    # --- 优雅的可视化渲染 ---
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))
    fig.suptitle(f"Pre-loaded Universal Gripper Mask (Frame {ref_idx})", fontsize=14, fontweight='bold')

    blended = ref_img.copy()
    blended[final_mask] = [50, 255, 50]  # 亮绿色高亮
    blended = cv2.addWeighted(ref_img, 0.6, blended, 0.4, 0)

    ax.imshow(blended)
    ax.axis('off')

    plt.tight_layout()
    plt.show()

    print("✅ 成功展示预加载的夹爪静态遮罩，完美跳过沉重的 SAM 推断！")
    return scene_constants

# ================= 🚀 一键启动 =================
scene_constants = visualize_preloaded_gripper_mask(scene_constants)

In [ ]:
# @title 🎯 展示预加载的夹爪表面深度 (跳过时序积分版)

def render_distilled_gripper_3d(median_depth, K_mat, rgb_img):
    v, u = np.where(median_depth > 0)
    z = median_depth[v, u]
    x = (u - K_mat[0, 2]) * z / K_mat[0, 0]
    y = (v - K_mat[1, 2]) * z / K_mat[1, 1]

    pts_3d = np.stack([x, y, z], axis=-1)
    fig = go.Figure(data=[go.Scatter3d(
        x=pts_3d[:, 0], y=pts_3d[:, 1], z=pts_3d[:, 2],
        mode='markers', marker=dict(size=2, color=rgb_img[v, u], opacity=0.8)
    )])
    fig.update_layout(
        title="Distilled Gripper Surface 🦾",
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Depth (Z)', aspectmode='data',
                   camera=dict(eye=dict(x=0, y=-0.5, z=-1.5), up=dict(x=0, y=-1, z=0))),
        margin=dict(l=0, r=0, b=0, t=40)
    )
    fig.show()

def visualize_preloaded_gripper_depth(scene_constants):
    wrist_cam = scene_constants['meta']['wrist_serial']
    cam_data = scene_constants['camera'][wrist_cam]

    if 'empirical_gripper_depth' not in cam_data:
        print("⚠️ 未找到预加载的 empirical_gripper_depth，请确保数据加载步骤成功读取了 gripper_depth.npz！")
        return scene_constants

    print("⚡ 极速模式：检测到预加载的极品夹爪深度，完美跳过数十万像素的时序积分计算！")
    median_depth_map = cam_data['empirical_gripper_depth']

    # 寻找第一帧完全闭合的画面作为底图
    gripper_states = scene_constants['robot']['gripper_positions']
    closed_indices = np.where(gripper_states < 0.05)[0]
    ref_idx = closed_indices[0] if len(closed_indices) > 0 else 0
    ref_rgb = cam_data['video_rgb'][ref_idx]

    # --- 2D 可视化验证大盘 (精简为 1x2) ---
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].imshow(ref_rgb)
    axes[0].set_title(f"Reference RGB (Frame {ref_idx})")
    axes[0].axis('off')

    im = axes[1].imshow(median_depth_map, cmap='plasma')
    axes[1].set_title("Pre-loaded Distilled Median Depth")
    axes[1].axis('off')
    fig.colorbar(im, ax=axes[1], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

    # --- 3D 升维验证 ---
    print("🚀 正在渲染预加载的 3D 点云表面...")
    # 注意：确保前面的单元格中已经定义了 render_distilled_gripper_3d 函数
    render_distilled_gripper_3d(median_depth_map, cam_data['K_mat'], ref_rgb)

    return scene_constants

# ================= 🚀 一键启动 =================
scene_constants = visualize_preloaded_gripper_depth(scene_constants)

### 相机外參标定

In [ ]:
# @title 3D 单帧融合点云与交互式可视化

def render_fused_point_cloud(scene_constants, scene_state, frame_idx=0,
                             max_render_points=150000, eye_pos=(-1.2, -1.2, 0.8), use_tint=False):

    # 🌟 1. 拥抱新架构：直接提取 camera 层级的 key，并利用 sorted 保证多机位合并顺序的绝对一致性
    camera_ids = sorted(scene_constants['camera'].keys())

    # 🌟 2. 调色板保留：严格保留原版染色矩阵 (绿、红、蓝)，用于 Debug 模式下区分机位
    tint_colors = np.array([[0, 50, 0], [50, 0, 0], [0, 0, 50]])

    fused_points, fused_colors = [], []

    # 🌟 3. 语义复苏遍历：使用清晰的命名
    for idx, cam_id in enumerate(camera_ids):
        cam_data = scene_constants['camera'][cam_id]
        cam_state = scene_state[cam_id]

        # 🌟 直接提取原始深度图 (彻底剥离 scale 和 shift)
        raw_depth = cam_data['raw_depth'][frame_idx].astype(np.float32)

        # 🌟 直接传入 raw_depth 提取 3D 点 (依赖 unproject_to_3d 内部的深度范围截断)
        points_3d, colors_rgb = unproject_to_3d(
            raw_depth,
            cam_data['video_rgb'][frame_idx],
            cam_data['K_mat'],
            T_cam2world=cam_state['extrinsics'][frame_idx]
        )

        # 染色处理
        if use_tint:
            colors_rgb = np.clip(colors_rgb.astype(int) + tint_colors[idx % len(tint_colors)], 0, 255).astype(np.uint8)

        fused_points.append(points_3d)
        fused_colors.append(colors_rgb)

    # 🌟 4. 极致清爽的堆叠与渲染
    show_plotly_point_cloud(
        pts=np.vstack(fused_points),
        cols=np.vstack(fused_colors),
        title=f"Fused Point Cloud (Frame {frame_idx})" + (" 🎨 [Tinted Debug Mode]" if use_tint else ""),
        max_points=max_render_points,
        eye_pos=eye_pos
    )

In [ ]:
# @title 2D Mask 全视场联合体检函数

import inspect

def render_multiview_mask_inspection(scene_constants, scene_state, pb_renderer, frame_idx=0):
    """
    全视场三联屏监视器 (支持新老两代 Renderer 无缝切换 + 数据结构极致解耦版)
    """
    camera_ids = list(scene_constants['camera'].keys())
    wrist_cam = scene_constants['meta']['wrist_serial']

    # 🌟 1. 提取姿态数据
    joint_angles = scene_constants['robot']['joint_positions'][frame_idx]
    gripper_state = scene_constants['robot']['gripper_positions'][frame_idx]

    # 🌟 2. 智能路由 A：检测引擎是否支持控制夹爪
    sig = inspect.signature(pb_renderer.update_robot_pose)
    pb_renderer.update_robot_pose(joint_angles, gripper_state=gripper_state)

    # 开启多联屏画布
    fig, axes = plt.subplots(1, len(camera_ids), figsize=(12, 3))
    if len(camera_ids) == 1: axes = [axes]

    fig.suptitle(f"Multi-View Segmentation Mask Inspection (Frame {frame_idx})",
                 fontsize=20, fontweight='bold', y=1.05)

    for i, cam_id in enumerate(camera_ids):

        # 🌟 核心进化：彻底干掉 if/else 特判！
        # 无论你是环境相机还是腕部相机，直接从统一大盘中抽出当前帧的 4x4 绝对位姿
        extrinsics = scene_state[cam_id]['extrinsics'][frame_idx]

        intrinsics = scene_constants['camera'][cam_id]['K_mat']
        img_rgb = scene_constants['camera'][cam_id]['video_rgb'][frame_idx].copy()
        h_img, w_img = img_rgb.shape[:2]

        # 🌟 3. 智能路由 B：检测引擎用的是哪个渲染方法
        robot_mask = pb_renderer.render_mask(extrinsics, intrinsics, w_img, h_img) > 0

        # 亮绿色半透明叠加
        overlay = img_rgb.copy()
        overlay[robot_mask] = [50, 255, 50]
        blended_img = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

        # 绘制
        cam_type = "Wrist Camera" if cam_id == wrist_cam else "External Camera"
        axes[i].imshow(blended_img)
        axes[i].set_title(f"[{cam_type}]\nCam ID: {cam_id}", fontsize=15)
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# @title 通用数字孪生影棚

import importlib.util
import numpy as np
import pybullet as p
import pybullet_data

# ==========================================
# 🌟 2. 极致美学的双生替身物理影棚 (完美镜像接骨版)
# ==========================================
class PyBulletRenderer_Robotiq:
    def __init__(self, ghost_urdf="PointWorld/assets/franka_description/franka_panda_robotiq_2f85_og.urdf"):
        """🌟 极致美学的双生替身物理影棚 (满血修复物理连杆与左右镜像)"""
        if p.isConnected(): p.disconnect()
        p.connect(p.DIRECT)
        p.setAdditionalSearchPath(pybullet_data.getDataPath())

        if importlib.util.find_spec('eglRendererPlugin'):
            p.loadPlugin(importlib.util.find_spec('eglRendererPlugin').origin, "_eglRendererPlugin")

        # ------------------------------------------
        # 实体 (The Body)：原厂极瘦机械臂
        # ------------------------------------------
        self.robot_id = p.loadURDF("franka_panda/panda.urdf", useFixedBase=True)
        self.arm_joints = [i for i in range(p.getNumJoints(self.robot_id)) if "panda_joint" in p.getJointInfo(self.robot_id, i)[1].decode('utf-8') and p.getJointInfo(self.robot_id, i)[2] != p.JOINT_FIXED]

        self.hidden_robot_links = []
        for i in range(-1, p.getNumJoints(self.robot_id)):
            name = p.getBodyInfo(self.robot_id)[0].decode('utf-8') if i == -1 else p.getJointInfo(self.robot_id, i)[12].decode('utf-8')
            if "hand" in name or "finger" in name:
                p.changeVisualShape(self.robot_id, i, rgbaColor=[0, 0, 0, 0])
                self.hidden_robot_links.append(i)

        # ------------------------------------------
        # 替身 (The Ghost)：携带夹爪的 PointWorld 机械臂
        # ------------------------------------------
        self.ghost_id = p.loadURDF(ghost_urdf, useFixedBase=True)
        self.ghost_arm_joints = [i for i in range(p.getNumJoints(self.ghost_id)) if "panda_joint" in p.getJointInfo(self.ghost_id, i)[1].decode('utf-8') and p.getJointInfo(self.ghost_id, i)[2] != p.JOINT_FIXED]

        # 🌟 修复核心：精确识别连杆层级与左右侧运动学镜像
        self.gripper_joints = []
        self.gripper_signs = []

        for i in range(p.getNumJoints(self.ghost_id)):
            info = p.getJointInfo(self.ghost_id, i)
            joint_name = info[1].decode('utf-8')
            joint_type = info[2]

            if joint_type != p.JOINT_FIXED and "panda_joint" not in joint_name:
                self.gripper_joints.append(i)

                # 🌟 骨科手术 2.0：应对右侧手指的反向坐标轴
                base_sign = -1 if "right" in joint_name else 1

                # 💥 致命穿模 Bug 修复：加入了 "finger_tip"，保持平行四边形机构！
                if "inner_finger" in joint_name or "follower" in joint_name or "finger_tip" in joint_name:
                    self.gripper_signs.append(base_sign * -1)
                else:
                    self.gripper_signs.append(base_sign)

        self.hidden_ghost_links = []
        for i in range(-1, p.getNumJoints(self.ghost_id)):
            name = p.getBodyInfo(self.ghost_id)[0].decode('utf-8') if i == -1 else p.getJointInfo(self.ghost_id, i)[12].decode('utf-8')
            if "panda_link" in name:
                p.changeVisualShape(self.ghost_id, i, rgbaColor=[0, 0, 0, 0])
                self.hidden_ghost_links.append(i)

    def _get_projection_matrix(self, intrinsics, width, height):
        fx, fy, cx, cy = intrinsics[0, 0], intrinsics[1, 1], intrinsics[0, 2], intrinsics[1, 2]
        near, far = 0.01, 10.0
        return [2.0 * fx / width, 0.0, 0.0, 0.0,
                0.0, 2.0 * fy / height, 0.0, 0.0,
                1.0 - 2.0 * cx / width, 2.0 * cy / height - 1.0, (far + near) / (near - far), -1.0,
                0.0, 0.0, 2.0 * far * near / (near - far), 0.0]

    def update_robot_pose(self, joint_angles, gripper_state=None, gripper_width_offset=0.08):
        for i, angle in zip(self.arm_joints, joint_angles):
            p.resetJointState(self.robot_id, i, angle)
        for i, angle in zip(self.ghost_arm_joints, joint_angles):
            p.resetJointState(self.ghost_id, i, angle)

        if gripper_state is not None and len(self.gripper_joints) > 0:
            raw_val = gripper_state[0] if isinstance(gripper_state, (list, np.ndarray)) else gripper_state

            # 🌟 回归正道的正向映射！
            # DROID 数据: 0 (完全张开) -> 1 (完全闭合)
            # Robotiq 85: 0 (完全张开) -> 0.8028 (完全闭合)
            raw_val = np.clip(raw_val, 0.0, 1.0)
            max_urdf_radian = 0.8028

            # 🌟 核心：强行减去 offset，突破原厂物理极限让夹爪外扩
            angle = (raw_val * max_urdf_radian) - gripper_width_offset

            for i, sign in zip(self.gripper_joints, self.gripper_signs):
                p.resetJointState(self.ghost_id, i, angle * sign)

        # 🌟 强制刷新渲染网格
        p.performCollisionDetection()

    def render_depth(self, extrinsics, intrinsics, width, height):
        cam_pos, target_pos = extrinsics[:3, 3], extrinsics[:3, 3] + extrinsics[:3, 2]
        view_matrix = p.computeViewMatrix(cam_pos, target_pos, -extrinsics[:3, 1])
        proj_matrix = self._get_projection_matrix(intrinsics, width, height)
        _, _, _, depth_buffer, _ = p.getCameraImage(
            width, height, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
            renderer=p.ER_BULLET_HARDWARE_OPENGL
        )
        metric_depth = 0.1 / (10.0 - 9.99 * np.reshape(depth_buffer, (height, width)))
        return np.where(metric_depth < 9.9, metric_depth, 0.0)

    def render_mask(self, extrinsics, intrinsics, width, height):
        cam_pos, target_pos = extrinsics[:3, 3], extrinsics[:3, 3] + extrinsics[:3, 2]
        view_matrix = p.computeViewMatrix(cam_pos, target_pos, -extrinsics[:3, 1])
        proj_matrix = self._get_projection_matrix(intrinsics, width, height)
        _, _, _, _, seg_buffer = p.getCameraImage(
            width, height, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
            renderer=p.ER_BULLET_HARDWARE_OPENGL, flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX
        )
        seg_array = np.reshape(seg_buffer, (height, width)).astype(np.int32)
        obj_ids, link_ids = seg_array & 0xFFFFFF, (seg_array >> 24) - 1
        valid_robot = (obj_ids == self.robot_id) & ~np.isin(link_ids, self.hidden_robot_links)
        valid_ghost = (obj_ids == self.ghost_id) & ~np.isin(link_ids, self.hidden_ghost_links)
        return valid_robot | valid_ghost

In [ ]:
# @title 第零阶段：读取数据集外參（如果存在的话）

def make_4x4(vec_6d):
    """将 6DoF 向量 [x, y, z, rx, ry, rz] 优雅转换为 4x4 齐次变换矩阵"""
    transform = np.eye(4)
    transform[:3, :3] = R.from_euler('xyz', vec_6d[3:]).as_matrix()
    transform[:3, 3] = vec_6d[:3]
    return transform

def init_camera_states(scene_constants, extrinsics_db):
    """组装多机位初始 3D 物理状态 (全局字典传参 + 内部安全路由版)"""
    print("  🌐 初始化全场相机 3D 物理状态...")
    wrist_serial = scene_constants['meta']['wrist_serial']
    robot_data = scene_constants['robot']
    n_frames = len(robot_data['T_ee_base_all'])

    # 🌟 内部安全提取：直接从全局字典中摸取当前 Episode 的外参
    # 如果该 Episode 是那 5 万个没有外参的数据之一，则优雅地返回空字典 {}
    episode_id = scene_constants['meta']['episode_id']
    episode_extrinsics = extrinsics_db.get(episode_id, {})

    scene_state = {}

    # 🌟 纯粹的遍历：直接迭代 camera 层级的 keys，彻底干掉丑陋的 if 过滤
    for cam_id in scene_constants['camera'].keys():

        if cam_id == wrist_serial:
            # 🦾 腕部相机：不随时间变化的量是【初始手眼标定矩阵】(4x4)
            base_ext = robot_data['T_cam_ee_init']
            cam_trajectory = robot_data['T_ee_base_all'] @ base_ext
        elif cam_id in episode_extrinsics:
            ext_data = episode_extrinsics[cam_id]
            ext_vec = ext_data.get('extrinsics', ext_data) if isinstance(ext_data, dict) else ext_data

            # 📷 环境相机：不随时间变化的量是【静态外参矩阵】(4x4)
            base_ext = make_4x4(ext_vec)
            cam_trajectory = np.tile(base_ext, (n_frames, 1, 1))
        else:
            print(f"    ⚠️ 未找到环境相机 [{cam_id}] 的预标定外参，初始化为 None。")
            base_ext = None
            cam_trajectory = None

        # 🌟 极致清爽的字典组装：既保存物理本质(base_extrinsic)，又保留供渲染器调用的时序轨迹(extrinsics)
        scene_state[cam_id] = {
            'base_extrinsic': base_ext,     # <--- 核心新增：提纯出来的 4x4 静态变量，专门喂给后续的优化器
            'extrinsics': cam_trajectory,   # 保留 (N, 4, 4) 轨迹，保证下游渲染和重投影代码无需修改
        }

    return scene_state

# 🌟 调用时直接传入整个全局 extrinsics_db，彻底告别 KeyError！
init_scene_state = init_camera_states(scene_constants, extrinsics_db)

# 🌟 智能路由：检查是否所有相机的初始外参都已存在
all_extrinsics_exist = all(state['extrinsics'] is not None for state in init_scene_state.values())

if all_extrinsics_exist:
    # 统一使用 init_scene_state 进行后续渲染（哪怕外参是 None，后续有保护机制就不会崩）
    render_fused_point_cloud(
        scene_constants=scene_constants,
        scene_state=init_scene_state,
        use_tint=True
    )

    pb_renderer_ultimate = PyBulletRenderer_Robotiq()

    render_multiview_mask_inspection(
        scene_constants=scene_constants,
        scene_state=init_scene_state,
        pb_renderer=pb_renderer_ultimate,
    )
else:
  print("  ✅ 没有检测到官方完整的外参真值，跳过 init_scene_state 渲染！")

In [ ]:
# @title 第一阶段：VGGT 视觉物理链式锚定 (仅使用第1帧极速静态版)

def vggt_warmup_extrinsics(scene_constants): # 🗑️ 彻底砍掉 scene_state 参数输入
    print(f"\n🌍 启动 VGGT 物理锚定：仅抽取第1帧推导绝对位姿...")
    wrist_serial = scene_constants['meta']['wrist_serial']
    ext_cams = [cam for cam in scene_constants['camera'].keys() if cam != wrist_serial]

    ref_cam = ext_cams[0]
    other_cams = ext_cams[1:]

    # 🌟 1. 破茧重生：直接在这里初始化全新的全局状态大盘！
    new_scene_state = {}
    robot_data = scene_constants['robot']
    n_frames_total = len(scene_constants['camera'][ref_cam]['video_rgb'])

    # 🌟 2. 顺手把腕部相机 (Wrist) 的全时序运动学轨迹算出来，奠定物理基石
    new_scene_state[wrist_serial] = {
        'base_extrinsic': robot_data['T_cam_ee_init'], # <--- 新增：保存静态手眼矩阵作为基底
        'extrinsics': robot_data['T_ee_base_all'] @ robot_data['T_cam_ee_init'],
    }

    # === 取消循环，仅提取第1帧 [0] 进行全量推理 ===
    print("  📸 正在提取多视角图像的第1帧并送入大模型...")

    # 1. 组装输入序列: [Ref] + [Other_Exts...] + [Wrist] (全部取第1帧 [0])
    img_ref = scene_constants['camera'][ref_cam]['video_rgb'][0]
    img_others = [scene_constants['camera'][cam]['video_rgb'][0] for cam in other_cams]
    img_wrist = scene_constants['camera'][wrist_serial]['video_rgb'][0]

    img_list = [img_ref] + img_others + [img_wrist]

    # 2. 一次推理，拿到所有相对位姿！
    T_rel_list = estimate_multi_camera_vggt(img_list)

    T_ref_to_others = T_rel_list[:-1]
    T_ref_to_wrist = T_rel_list[-1]

    # 3. 获取腕部相机在第1帧的绝对物理位姿 (GT)
    T_ee_base_first = scene_constants['robot']['T_ee_base_all'][0]
    T_cam_ee = scene_constants['robot']['T_cam_ee_init']
    T_wrist_to_base_first = T_ee_base_first @ T_cam_ee

    # 4. 终极链式法则解算绝对位姿
    T_ref_to_base = T_wrist_to_base_first @ T_ref_to_wrist

    # 🌟 3. 将计算出的静态外参直接平铺压入全新的状态字典 (同时剥离并保存 4x4 的 base_extrinsic)
    new_scene_state[ref_cam] = {
        'base_extrinsic': T_ref_to_base, # <--- 新增：保存纯粹的静态基础位姿
        'extrinsics': np.tile(T_ref_to_base, (n_frames_total, 1, 1)),
    }

    for tgt_cam, T_ref_to_tgt in zip(other_cams, T_ref_to_others):
        T_tgt_to_base = T_ref_to_base @ np.linalg.inv(T_ref_to_tgt)
        new_scene_state[tgt_cam] = {
            'base_extrinsic': T_tgt_to_base, # <--- 新增：保存纯粹的静态基础位姿
            'extrinsics': np.tile(T_tgt_to_base, (n_frames_total, 1, 1)),
        }

    print("  ✅ 极速物理锚定完成！")
    return new_scene_state

# 🌟 2. 核心拦截：在组装 3D 状态之前，用 VGGT 链式法则彻底洗掉垃圾外参！
vggt_scene_state = vggt_warmup_extrinsics(scene_constants)

In [ ]:
# @title 第二阶段：外部相机-机械臂本体精准对齐 (GPU 全局批处理终极美学版)

def axis_angle_to_matrix(rot_vec):
    """优雅可导的 Rodrigues 旋转公式 (引入 torch.where 极简分支)"""
    theta2 = torch.sum(rot_vec ** 2)
    theta = torch.sqrt(theta2 + 1e-16)
    k = rot_vec / theta

    # 利用反对称矩阵特性，直接进行优雅赋值
    K = torch.zeros((3, 3), device=rot_vec.device)
    K[0, 1], K[0, 2] = -k[2], k[1]
    K[1, 0], K[1, 2] =  k[2], -k[0]
    K[2, 0], K[2, 1] = -k[1],  k[0]

    R_exact = torch.eye(3, device=rot_vec.device) + torch.sin(theta)*K + (1-torch.cos(theta))*torch.mm(K, K)

    # 泰勒展开近似 (应对极小角度的奇异点)
    K_approx = torch.zeros_like(K)
    K_approx[0, 1], K_approx[0, 2] = -rot_vec[2], rot_vec[1]
    K_approx[1, 0], K_approx[1, 2] =  rot_vec[2], -rot_vec[0]
    K_approx[2, 0], K_approx[2, 1] = -rot_vec[1],  rot_vec[0]
    R_approx = torch.eye(3, device=rot_vec.device) + K_approx

    # 🌟 极简张量条件分支，抛弃算术遮罩
    return torch.where(theta2 < 1e-8, R_approx, R_exact)

def get_foreground_robot_points(T_init, K, obs_depth, pb_renderer, max_pts):
    """极速机器人点云提取 (引入 Pythonic 的采样布尔魔法)"""
    h_img, w_img = obs_depth.shape
    render_d = pb_renderer.render_depth(T_init, K, w_img, h_img)

    v_r, u_r = np.where(render_d > 0)
    z_r = render_d[v_r, u_r]

    if len(z_r) < max_pts: return None

    P_cam_r = np.stack([(u_r - K[0, 2]) * z_r / K[0, 0], (v_r - K[1, 2]) * z_r / K[1, 1], z_r, np.ones_like(z_r)])
    pts_robot_world = (T_init @ P_cam_r)[:3, :].T

    # 🌟 绝杀！利用动态布尔值 replace=(...)，一行代码搞定所有逻辑判定
    idx = np.random.choice(len(pts_robot_world), max_pts, replace=(len(pts_robot_world) < max_pts))
    return torch.tensor(pts_robot_world[idx], dtype=torch.float32, device=device)

def compute_robot_loss_batched(batch_X, T_opt, K, batch_obs):
    """榨干 GPU 算力的全局 Batched 纯水管算子 (无中间冗余变量)"""
    B, _, h_img, w_img = batch_obs.shape

    P_c = (batch_X - T_opt[:3, 3]) @ T_opt[:3, :3]
    Z_pred = P_c[..., 2]

    u = K[0, 0] * P_c[..., 0] / Z_pred + K[0, 2]
    v = K[1, 1] * P_c[..., 1] / Z_pred + K[1, 2]

    # 🌟 一气呵成：计算归一化坐标并直接堆叠送入网格
    grid = torch.stack([(u / (w_img - 1)) * 2 - 1, (v / (h_img - 1)) * 2 - 1], dim=-1).unsqueeze(1)
    Z_obs_raw = F.grid_sample(batch_obs, grid, mode='bilinear', padding_mode='border', align_corners=True).squeeze(1).squeeze(1)

    # 🌟 大一统联合物理遮罩：将内外边界判定全部融为一体
    valid_mask = (Z_pred > 0.) & (Z_pred < 1.5) & (Z_obs_raw > 0.) & (Z_obs_raw < 1.5) & \
                 (u >= 0) & (u < w_img - 1) & (v >= 0) & (v < h_img - 1)

    diff = torch.abs(Z_obs_raw[valid_mask] - Z_pred[valid_mask])
    return torch.nan_to_num(diff.mean(), nan=0.0)

# ================= 第一阶段主循环 =================

def run_stage2_robot_alignment(scene_constants, pb_renderer, init_scene_state=None, vggt_scene_state=None):
    OUTER_LOOPS = 5
    INNER_LOOPS = 100
    MAX_ROBOT_PTS = 2000

    print(f"\n🦾 启动第一阶段优化 (GPU 全局批处理终极美学版 | 双基底优选模式)...")

    ext_cams = [c for c in scene_constants['camera'].keys() if c != scene_constants['meta']['wrist_serial']]
    n_frames = len(scene_constants['camera'][ext_cams[0]]['video_rgb'])

    def make_T(delta):
        T = torch.eye(4, device=device)
        T[:3, :3], T[:3, 3] = axis_angle_to_matrix(delta[:3]), delta[3:]
        return T

    # 判断是否拥有官方预标定外参
    has_init = init_scene_state is not None

    # 初始化一个最终状态池，默认以 vggt 作为保底格式基础
    pybullet_scene_state = copy.deepcopy(init_scene_state if has_init else vggt_scene_state)

    # 🌟 提取核心优化逻辑为闭包，方便传入不同的基底重复调用
    def optimize_camera_with_base(cam_id, base_extrinsic):
        T_init_t = torch.tensor(base_extrinsic, dtype=torch.float32, device=device)
        K_t = torch.tensor(scene_constants['camera'][cam_id]['K_mat'], dtype=torch.float32, device=device)
        K_np = scene_constants['camera'][cam_id]['K_mat']

        d_ext = torch.zeros(6, requires_grad=True, device=device)
        optimizer = optim.Adam([d_ext], lr=0.001)

        final_loss = float('inf')

        for outer_step in range(OUTER_LOOPS):
            with torch.no_grad():
                T_cur_np = (T_init_t @ make_T(d_ext)).cpu().numpy()

            cache_X, cache_obs = [], []
            for t in range(n_frames):
                pb_renderer.update_robot_pose(scene_constants['robot']['joint_positions'][t])
                d_obs = scene_constants['camera'][cam_id]['raw_depth'][t].astype(np.float32)
                r_pts_t = get_foreground_robot_points(T_cur_np, K_np, d_obs, pb_renderer, MAX_ROBOT_PTS)

                if r_pts_t is not None:
                    cache_X.append(r_pts_t)
                    cache_obs.append(torch.tensor(d_obs, dtype=torch.float32, device=device)[None, ...])

            if not cache_X:
                continue

            batch_X = torch.stack(cache_X)
            batch_obs = torch.stack(cache_obs)

            for inner_step in range(INNER_LOOPS):
                optimizer.zero_grad()
                loss_rob = compute_robot_loss_batched(batch_X, T_init_t @ make_T(d_ext), K_t, batch_obs)
                loss_rob.backward()
                optimizer.step()
                final_loss = loss_rob.item()

                if inner_step % 50 == 0 or inner_step == INNER_LOOPS - 1:
                    print(f"      Outer {outer_step+1}/{OUTER_LOOPS} | Inner {inner_step:03d} | Robot Loss: {final_loss:.4f}")

        # 计算并返回优化结果
        with torch.no_grad():
            T_final_np = (T_init_t @ make_T(d_ext)).cpu().numpy()
            shift_mm = np.linalg.norm(d_ext[3:].detach().cpu().numpy()) * 1000

        return T_final_np, final_loss, shift_mm

    # 开始逐相机优化
    for cam in ext_cams:
        print(f"\n  📷 正在独立优化外部相机: [{cam}] ...")

        # 准备待选池
        base_candidates = [("VGGT", vggt_scene_state[cam]['base_extrinsic'])]
        if has_init:
            base_candidates.append(("Dataset Init", init_scene_state[cam]['base_extrinsic']))

        best_loss = float('inf')
        best_T = None
        best_source_name = None
        best_shift = 0.0

        for source_name, base_ext in base_candidates:
            print(f"    --> 🔄 尝试基于 [{source_name}] 基底进行解算:")
            T_res, loss_res, shift_res = optimize_camera_with_base(cam, base_ext)

            if loss_res < best_loss:
                best_loss = loss_res
                best_T = T_res
                best_source_name = source_name
                best_shift = shift_res

        print(f"  ✅ [{cam}] 刚体对齐完毕！采纳最优基底: {best_source_name}, 最终 Loss: {best_loss:.4f} (相对修正: {best_shift:.2f}mm)")

        # 写入最终大盘
        pybullet_scene_state[cam]['base_extrinsic'] = best_T
        pybullet_scene_state[cam]['extrinsics'] = np.tile(best_T, (n_frames, 1, 1))

    return pybullet_scene_state

# ================= 启动调用与渲染展示 =================

all_extrinsics_exist = all(state['extrinsics'] is not None for state in init_scene_state.values())

pb_renderer_ultimate = PyBulletRenderer_Robotiq()

# 🌟 传入两个状态让函数自己去打擂台
pybullet_scene_state = run_stage2_robot_alignment(
    scene_constants=scene_constants,
    pb_renderer=pb_renderer_ultimate,
    init_scene_state=init_scene_state if all_extrinsics_exist else None,
    vggt_scene_state=vggt_scene_state
)

render_fused_point_cloud(
    scene_constants=scene_constants,
    scene_state=pybullet_scene_state,
    use_tint=True
)

render_multiview_mask_inspection(
    scene_constants=scene_constants,
    scene_state=pybullet_scene_state,
    pb_renderer=pb_renderer_ultimate,
)

In [ ]:
# @title 第二阶段：腕部相机-夹爪本体精准对齐 (GPU 专属定制版)

def get_foreground_gripper_points(T_cam_world, K, obs_depth, pb_renderer, max_pts):
    """🔪 精准手术刀：只提取夹爪本体的点云，彻底剔除机械臂连杆"""
    h_img, w_img = obs_depth.shape
    cam_pos, target_pos = T_cam_world[:3, 3], T_cam_world[:3, 3] + T_cam_world[:3, 2]

    view_matrix = p.computeViewMatrix(cam_pos, target_pos, -T_cam_world[:3, 1])
    proj_matrix = pb_renderer._get_projection_matrix(K, w_img, h_img)

    # 🌟 调用底层接口，同时提取深度与语义分割掩码
    _, _, _, depth_buffer, seg_buffer = p.getCameraImage(
        w_img, h_img, viewMatrix=view_matrix, projectionMatrix=proj_matrix,
        renderer=p.ER_BULLET_HARDWARE_OPENGL, flags=p.ER_SEGMENTATION_MASK_OBJECT_AND_LINKINDEX
    )

    metric_depth = 0.1 / (10.0 - 9.99 * np.reshape(depth_buffer, (h_img, w_img)))
    seg_array = np.reshape(seg_buffer, (h_img, w_img)).astype(np.int32)
    obj_ids = seg_array & 0xFFFFFF

    # 🌟 核心过滤：仅仅保留代表夹爪的 ghost_id 对应的像素
    valid_ghost = (obj_ids == pb_renderer.ghost_id)
    v_r, u_r = np.where((metric_depth < 9.9) & valid_ghost)
    z_r = metric_depth[v_r, u_r]

    if len(z_r) < 100: return None

    # 获取在相机坐标系下的齐次坐标
    P_cam_r = np.stack([(u_r - K[0, 2]) * z_r / K[0, 0], (v_r - K[1, 2]) * z_r / K[1, 1], z_r, np.ones_like(z_r)])

    # 随机降采样以喂入 GPU
    idx = np.random.choice(len(z_r), max_pts, replace=(len(z_r) < max_pts))
    return P_cam_r[:, idx]

def compute_wrist_loss_batched(batch_P_ee, T_cam_ee_opt, K, batch_obs):
    """🦇 Wrist 专属算子：通过优化中的手眼逆矩阵将末端点拨回相机系"""
    B, _, h_img, w_img = batch_obs.shape

    # 🌟 获取 T_ee_cam (相机到末端的逆变换)
    T_ee_cam = torch.linalg.inv(T_cam_ee_opt)

    # 将被锚定在末端坐标系(EE)的点，重新投影到当前的相机坐标系
    P_c = batch_P_ee @ T_ee_cam[:3, :3].T + T_ee_cam[:3, 3]
    Z_pred = P_c[..., 2]

    u = K[0, 0] * P_c[..., 0] / Z_pred + K[0, 2]
    v = K[1, 1] * P_c[..., 1] / Z_pred + K[1, 2]

    grid = torch.stack([(u / (w_img - 1)) * 2 - 1, (v / (h_img - 1)) * 2 - 1], dim=-1).unsqueeze(1)
    Z_obs_raw = F.grid_sample(batch_obs, grid, mode='bilinear', padding_mode='border', align_corners=True).squeeze(1).squeeze(1)

    valid_mask = (Z_pred > 0.) & (Z_pred < 1.5) & (Z_obs_raw > 0.) & (Z_obs_raw < 1.5) & \
                 (u >= 0) & (u < w_img - 1) & (v >= 0) & (v < h_img - 1)

    diff = torch.abs(Z_obs_raw[valid_mask] - Z_pred[valid_mask])
    return torch.nan_to_num(diff.mean(), nan=0.0)

# ================= 腕部专属对齐引擎 =================

def run_stage2_wrist_alignment(scene_constants, init_scene_state, pb_renderer):
    OUTER_LOOPS = 5
    INNER_LOOPS = 100
    MAX_ROBOT_PTS = 2000

    print(f"\n🦾 启动腕部实验：纯夹爪物理锚定 (仅优化 Wrist Camera)...")

    wrist_cam = scene_constants['meta']['wrist_serial']
    n_frames = len(scene_constants['camera'][wrist_cam]['video_rgb'])

    pybullet_scene_state = copy.deepcopy(init_scene_state)
    frame_point_counts = {t: 0 for t in range(n_frames)}

    # 提取初始的手眼矩阵 (T_cam_ee_init)
    T_cam_ee_init_np = init_scene_state[wrist_cam]['base_extrinsic']
    T_cam_ee_init_t = torch.tensor(T_cam_ee_init_np, dtype=torch.float32, device=device)

    K_np = scene_constants['camera'][wrist_cam]['K_mat']
    K_t = torch.tensor(K_np, dtype=torch.float32, device=device)

    T_ee_base_all = scene_constants['robot']['T_ee_base_all']

    d_ext = torch.zeros(6, requires_grad=True, device=device)
    optimizer = optim.Adam([d_ext], lr=0.001)

    def make_T(delta):
        T = torch.eye(4, device=device)
        T[:3, :3], T[:3, 3] = axis_angle_to_matrix(delta[:3]), delta[3:]
        return T

    for outer_step in range(OUTER_LOOPS):
        with torch.no_grad():
            T_cam_ee_np = (T_cam_ee_init_t @ make_T(d_ext)).cpu().numpy()

        cache_P_ee, cache_obs = [], []

        for t in range(n_frames):
            # 🌟 必须同步更新关节和夹爪状态
            pb_renderer.update_robot_pose(scene_constants['robot']['joint_positions'][t],
                                          gripper_state=scene_constants['robot']['gripper_positions'][t])

            T_cam_world_np = T_ee_base_all[t] @ T_cam_ee_np
            d_obs = scene_constants['camera'][wrist_cam]['raw_depth'][t].astype(np.float32)

            # 获取相机系下的纯夹爪点云
            P_cam_r = get_foreground_gripper_points(T_cam_world_np, K_np, d_obs, pb_renderer, MAX_ROBOT_PTS)

            if P_cam_r is not None:
                # 🌟 数据解耦锚定：将相机系的点云，通过当前手眼矩阵转移到末端坐标系(EE)锁定
                P_ee_r = (T_cam_ee_np @ P_cam_r)[:3, :].T

                frame_point_counts[t] += len(P_ee_r)
                cache_P_ee.append(torch.tensor(P_ee_r, dtype=torch.float32, device=device))
                cache_obs.append(torch.tensor(d_obs, dtype=torch.float32, device=device)[None, ...])

        if not cache_P_ee:
            print(f"    ⚠️ 未找到有效的夹爪点云！请检查渲染逻辑。")
            break

        batch_P_ee = torch.stack(cache_P_ee)
        batch_obs = torch.stack(cache_obs)

        for inner_step in range(INNER_LOOPS):
            optimizer.zero_grad()

            T_cam_ee_opt = T_cam_ee_init_t @ make_T(d_ext)
            loss_rob = compute_wrist_loss_batched(batch_P_ee, T_cam_ee_opt, K_t, batch_obs)
            loss_rob.backward()
            optimizer.step()

            # 🌟 核心修改：增加平移和旋转的改变量监控
            if inner_step % 50 == 0 or inner_step == INNER_LOOPS - 1:
                with torch.no_grad():
                    # 旋转向量的模长即为旋转的弧度，转换为角度
                    rot_deg = torch.norm(d_ext[:3]).item() * (180.0 / np.pi)
                    # 平移向量的模长即为平移量，转换为毫米
                    shift_mm = torch.norm(d_ext[3:]).item() * 1000.0
                print(f"    Outer {outer_step+1}/{OUTER_LOOPS} | Inner {inner_step:03d} | Wrist Loss: {loss_rob.item():.4f} | Shift: {shift_mm:.2f}mm | Rot: {rot_deg:.2f}°")

    with torch.no_grad():
        T_cam_ee_final = (T_cam_ee_init_t @ make_T(d_ext)).cpu().numpy()
        shift_mm = torch.norm(d_ext[3:]).item() * 1000.0
        rot_deg = torch.norm(d_ext[:3]).item() * (180.0 / np.pi)
        print(f"  ✅ [Wrist Cam: {wrist_cam}] 手眼标定收敛！纯夹爪对齐修正量 -> 平移: {shift_mm:.2f}mm, 旋转: {rot_deg:.2f}°")

        # 🌟 覆盖更新状态大盘
        pybullet_scene_state[wrist_cam]['base_extrinsic'] = T_cam_ee_final
        pybullet_scene_state[wrist_cam]['extrinsics'] = T_ee_base_all @ T_cam_ee_final

    return pybullet_scene_state

# ================= 启动验证 =================

pb_renderer_ultimate = PyBulletRenderer_Robotiq()

# 覆盖并获取专为 Wrist 优化的 scene_state
wrist_optimized_state = run_stage2_wrist_alignment(
    scene_constants=scene_constants,
    init_scene_state=pybullet_scene_state,
    pb_renderer=pb_renderer_ultimate
)

# 可视化渲染验证修改效果 (你可以指定一个你想看的手眼画面)
render_multiview_mask_inspection(
    scene_constants=scene_constants,
    scene_state=wrist_optimized_state,
    pb_renderer=pb_renderer_ultimate,
)

In [ ]:
# @title 第三阶段实验：全域环境缝合与机器人本体联合优化 (全机位大一统版)

def batched_chamfer_distance(p1, p2):
    """纯矩阵化并行倒角距离 (硬编码 5cm 物理截断：你的最后一道防线)"""
    dist_matrix = torch.cdist(p1, p2)
    min_dist_12 = torch.min(dist_matrix, dim=2)[0]
    min_dist_21 = torch.min(dist_matrix, dim=1)[0]

    valid_12 = min_dist_12 < 0.05
    valid_21 = min_dist_21 < 0.05

    loss = torch.tensor(0.0, device=device)
    if valid_12.any(): loss += min_dist_12[valid_12].mean()
    if valid_21.any(): loss += min_dist_21[valid_21].mean()

    overlap_ratio = (valid_12.sum() + valid_21.sum()) / (p1.shape[0] * (p1.shape[1] + p2.shape[1]) + 1e-6)
    return loss, overlap_ratio.item()

def get_cam_points_t(t, cam_data):
    """预计算算子：无差别提取视场内的所有点云 (包含桌面、墙壁、人体等全域环境)"""
    depth = cam_data['raw_depth'][t].astype(np.float32)
    K_mat_np = cam_data['K_mat']

    valid_mask = (depth > 0.) & (depth < 1.5)
    vs, us = np.where(valid_mask)
    if len(us) < 100: return None

    zs_obs = depth[vs, us]
    x_c = (us - K_mat_np[0, 2]) * zs_obs / K_mat_np[0, 0]
    y_c = (vs - K_mat_np[1, 2]) * zs_obs / K_mat_np[1, 1]

    # 构建相机系齐次坐标
    P_cam = np.stack([x_c, y_c, zs_obs, np.ones_like(zs_obs)], axis=0)

    # 💥 BBox 物理法则被彻底粉碎，所有 P_cam 直接进入采样池！
    if P_cam.shape[1] < 100: return None
    if P_cam.shape[1] > 2000:
        idx = np.random.choice(P_cam.shape[1], 2000, replace=False)
    else:
        idx = np.random.choice(P_cam.shape[1], 2000, replace=True)

    return torch.tensor(P_cam[:, idx], dtype=torch.float32, device=device)

# 🎯 核心算子：全域环境缝合与机器人本体联合优化引擎
def run_global_joint_alignment(scene_constants, prev_scene_state, pb_renderer, lr=0.001, robot_weight=1.0):
    print(f"\n🌍 大一统环境缝合与机器人本体联合优化 (lr={lr}, weight={robot_weight})...")
    camera_ids = list(scene_constants['camera'].keys())
    wrist_cam = scene_constants['meta']['wrist_serial']
    ext_cams = [c for c in camera_ids if c != wrist_cam]
    cam1, cam2 = ext_cams[0], ext_cams[1]
    n_frames = len(scene_constants['camera'][cam1]['video_rgb'])

    robot_joints_seq = scene_constants['robot']['joint_positions']
    gripper_states_seq = scene_constants['robot']['gripper_positions']

    def to_t(arr): return torch.tensor(arr, dtype=torch.float32, device=device)
    def make_T(delta):
        T = torch.eye(4, device=device)
        T[:3, :3], T[:3, 3] = axis_angle_to_matrix(delta[:3]), delta[3:]
        return T

    T_ee_all = scene_constants['robot']['T_ee_base_all']
    T_cam_ee_init = prev_scene_state[wrist_cam]['base_extrinsic']
    init_p1 = prev_scene_state[cam1]['base_extrinsic']
    init_p2 = prev_scene_state[cam2]['base_extrinsic']

    K_np1, K_np2, K_np_w = scene_constants['camera'][cam1]['K_mat'], scene_constants['camera'][cam2]['K_mat'], scene_constants['camera'][wrist_cam]['K_mat']
    K_t1, K_t2, K_t_w = to_t(K_np1), to_t(K_np2), to_t(K_np_w)

    cache_P1, cache_P2, cache_Pw, cache_Tee = [], [], [], []
    cache_X1, cache_obs1, cache_X2, cache_obs2 = [], [], [], []
    cache_P_ee, cache_obs_w = [], []

    print(f"  🔍 正在预计算 Chamfer 环境点云、外部 Robot 物理点云 与 腕部夹爪锚点...")
    for t in range(n_frames):
        p1 = get_cam_points_t(t, scene_constants['camera'][cam1])
        p2 = get_cam_points_t(t, scene_constants['camera'][cam2])
        pw = get_cam_points_t(t, scene_constants['camera'][wrist_cam])
        if p1 is not None and p2 is not None and pw is not None:
            cache_P1.append(p1); cache_P2.append(p2); cache_Pw.append(pw); cache_Tee.append(to_t(T_ee_all[t]))

        pb_renderer.update_robot_pose(robot_joints_seq[t], gripper_state=gripper_states_seq[t])

        d_obs1 = scene_constants['camera'][cam1]['raw_depth'][t].astype(np.float32)
        r_pts1 = get_foreground_robot_points(init_p1, K_np1, d_obs1, pb_renderer, max_pts=2000)
        if r_pts1 is not None:
            cache_X1.append(r_pts1); cache_obs1.append(torch.tensor(d_obs1, dtype=torch.float32, device=device)[None, ...])

        d_obs2 = scene_constants['camera'][cam2]['raw_depth'][t].astype(np.float32)
        r_pts2 = get_foreground_robot_points(init_p2, K_np2, d_obs2, pb_renderer, max_pts=2000)
        if r_pts2 is not None:
            cache_X2.append(r_pts2); cache_obs2.append(torch.tensor(d_obs2, dtype=torch.float32, device=device)[None, ...])

        T_cam_world_np = T_ee_all[t] @ T_cam_ee_init
        d_obs_w = scene_constants['camera'][wrist_cam]['raw_depth'][t].astype(np.float32)
        P_cam_r = get_foreground_gripper_points(T_cam_world_np, K_np_w, d_obs_w, pb_renderer, max_pts=2000)
        if P_cam_r is not None:
            P_ee_r = (T_cam_ee_init @ P_cam_r)[:3, :].T
            cache_P_ee.append(torch.tensor(P_ee_r, dtype=torch.float32, device=device))
            cache_obs_w.append(torch.tensor(d_obs_w, dtype=torch.float32, device=device)[None, ...])

    batch_P1, batch_P2, batch_Pw, batch_Tee = torch.stack(cache_P1), torch.stack(cache_P2), torch.stack(cache_Pw), torch.stack(cache_Tee)
    batch_X1, batch_obs1 = torch.stack(cache_X1), torch.stack(cache_obs1)
    batch_X2, batch_obs2 = torch.stack(cache_X2), torch.stack(cache_obs2)
    batch_P_ee = torch.stack(cache_P_ee) if cache_P_ee else None
    batch_obs_w = torch.stack(cache_obs_w) if cache_obs_w else None

    print(f"  ✅ 数据准备完毕！启动 GPU 大一统联合优化引擎 (Chamfer + Robot + Wrist)...")
    d1 = torch.zeros(6, requires_grad=True, device=device)
    d2 = torch.zeros(6, requires_grad=True, device=device)
    dhe = torch.zeros(6, requires_grad=True, device=device)

    # 🌟 动态注入学习率
    optimizer = optim.Adam([d1, d2, dhe], lr=lr)

    T1_init_t, T2_init_t, Tee_init_t = to_t(init_p1), to_t(init_p2), to_t(T_cam_ee_init)

    for step in range(500):
        optimizer.zero_grad()
        bc1 = (T1_init_t @ make_T(d1) @ batch_P1)[:, :3, :].transpose(1, 2)
        bc2 = (T2_init_t @ make_T(d2) @ batch_P2)[:, :3, :].transpose(1, 2)
        T_wrist_world = batch_Tee @ (Tee_init_t @ make_T(dhe))
        bcw = torch.bmm(T_wrist_world, batch_Pw)[:, :3, :].transpose(1, 2)

        l12, o12 = batched_chamfer_distance(bc1, bc2)
        l1w, o1w = batched_chamfer_distance(bc1, bcw)
        l2w, o2w = batched_chamfer_distance(bc2, bcw)
        loss_chamfer = l12 + l1w + l2w

        l_rob1 = compute_robot_loss_batched(batch_X1, T1_init_t @ make_T(d1), K_t1, batch_obs1)
        l_rob2 = compute_robot_loss_batched(batch_X2, T2_init_t @ make_T(d2), K_t2, batch_obs2)
        l_wrist = torch.tensor(0.0, device=device)
        if batch_P_ee is not None:
            l_wrist = compute_wrist_loss_batched(batch_P_ee, Tee_init_t @ make_T(dhe), K_t_w, batch_obs_w)

        # 🌟 动态注入物理 Loss 权重
        loss_total = loss_chamfer + robot_weight * (l_rob1 + l_rob2 + l_wrist)
        loss_total.backward()
        optimizer.step()

        if step % 50 == 0 or step == 500 - 1:
            bg_overlap = (o12 + o1w + o2w) / 3.0 * 100
            shift_c1, shift_c2, shift_w = torch.norm(d1[:3]).item() * 1000, torch.norm(d2[:3]).item() * 1000, torch.norm(dhe[:3]).item() * 1000
            print(f"    Step {step:03d} | Chmf: {loss_chamfer.item():.4f} | Rob1: {l_rob1.item():.4f} | Rob2: {l_rob2.item():.4f} | Wrst: {l_wrist.item():.4f} | BG Overlap: {bg_overlap:.1f}% | Shift -> C1: {shift_c1:.2f}mm, C2: {shift_c2:.2f}mm, W: {shift_w:.2f}mm")

    with torch.no_grad():
        final_p1, final_p2, final_cam_ee = (T1_init_t @ make_T(d1)).cpu().numpy(), (T2_init_t @ make_T(d2)).cpu().numpy(), (Tee_init_t @ make_T(dhe)).cpu().numpy()

    print(f"\n✅ 3D 大一统联合优化收官！全机位相互牵制下产生的协同位移：")
    print(f"  📷 [静态 {cam1}]: {np.linalg.norm(d1[:3].detach().cpu().numpy()) * 1000:.2f} mm")
    print(f"  📷 [静态 {cam2}]: {np.linalg.norm(d2[:3].detach().cpu().numpy()) * 1000:.2f} mm")
    print(f"  🦾 [运动 Wrist]: {np.linalg.norm(dhe[:3].detach().cpu().numpy()) * 1000:.2f} mm")

    ultimate_scene_state = {cam: {} for cam in camera_ids}
    ultimate_scene_state[cam1].update({'base_extrinsic': final_p1, 'extrinsics': np.tile(final_p1, (n_frames, 1, 1))})
    ultimate_scene_state[cam2].update({'base_extrinsic': final_p2, 'extrinsics': np.tile(final_p2, (n_frames, 1, 1))})
    ultimate_scene_state[wrist_cam].update({'base_extrinsic': final_cam_ee, 'extrinsics': T_ee_all @ final_cam_ee})

    return ultimate_scene_state

# ================= 启动调用与渲染展示 =================
pb_renderer_ultimate = PyBulletRenderer_Robotiq()

stage3_scene_state = run_global_joint_alignment(
    scene_constants, wrist_optimized_state, pb_renderer_ultimate,
    lr=0.001, robot_weight=1.0
)

render_fused_point_cloud(
    scene_constants=scene_constants,
    scene_state=stage3_scene_state,
    use_tint=True
)

render_multiview_mask_inspection(
    scene_constants=scene_constants,
    scene_state=stage3_scene_state,
    pb_renderer=pb_renderer_ultimate,
)

In [ ]:
# @title 第四阶段实验：全域环境缝合与机器人本体联合优化 (精细雕琢版)

pb_renderer_ultimate = PyBulletRenderer_Robotiq()

ultimate_scene_state = run_global_joint_alignment(
    scene_constants, stage3_scene_state, pb_renderer_ultimate,
    lr=0.0001, robot_weight=0.1
)

render_fused_point_cloud(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
    use_tint=True
)

render_multiview_mask_inspection(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
    pb_renderer=pb_renderer_ultimate,
)

In [ ]:
# @title 2D 机械臂分割可视化 (双极态极限对比验证版)

def inspect_gripper_extremes(
    scene_constants,
    scene_state,
    pb_renderer,
    tgt_width=1800
):
    """
    自动抽取夹爪 State 接近 0 和绝对值最大 的两帧，上下对比渲染
    """
    gripper_states = scene_constants['robot']['gripper_positions']

    # 🌟 1. 寻找两个极限帧的索引
    # 找绝对值最大的一帧 (通常是最大张开或最大抓取)
    idx_max = np.argmax(np.abs(gripper_states))
    val_max = gripper_states[idx_max]

    # 找最接近 0 的一帧 (通常是完全闭合或完全张开的另一个极限)
    idx_zero = np.argmin(np.abs(gripper_states))
    val_zero = gripper_states[idx_zero]

    print(f"🎯 锁定验证帧 [State ~ 0]: 第 {idx_zero} 帧 | 夹爪数据: {val_zero:.4f}")
    print(f"🎯 锁定验证帧 [State Max]: 第 {idx_max} 帧 | 夹爪数据: {val_max:.4f}")

    camera_ids = list(scene_constants['camera'].keys())
    wrist_serial = scene_constants['meta']['wrist_serial']

    # 🌟 2. 内部封装一个单帧渲染的闭包函数，避免代码重复
    def render_single_frame(frame_idx, gripper_val):
        current_joints = scene_constants['robot']['joint_positions'][frame_idx]
        pb_renderer.update_robot_pose(current_joints, gripper_state=gripper_val)

        frame_views = []
        for cam_id in camera_ids:
            cam_data = scene_constants['camera'][cam_id]
            cam_state = scene_state[cam_id]

            img_rgb = cam_data['video_rgb'][frame_idx].copy()
            h_img, w_img = img_rgb.shape[:2]

            # 渲染 Mask
            robot_mask = pb_renderer.render_mask(
                extrinsics=cam_state['extrinsics'][frame_idx],
                intrinsics=cam_data['K_mat'],
                width=w_img,
                height=h_img
            ) > 0

            # 赛博朋克风混色
            overlay = img_rgb.copy()
            overlay[robot_mask] = [50, 150, 255]
            blended_img = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

            # 文本与阴影绘制
            is_wrist = (cam_id == wrist_serial)
            cam_type = "Wrist Cam" if is_wrist else "Ext Cam"
            cv2.putText(blended_img, f"{cam_type} [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
            cv2.putText(blended_img, f"{cam_type} [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)

            frame_views.append(blended_img)

        # 横向拼接并缩放
        row_concat = np.concatenate(frame_views, axis=1)
        tgt_height = int(row_concat.shape[0] * (tgt_width / row_concat.shape[1]))
        return cv2.resize(row_concat, (tgt_width, tgt_height))

    # 🌟 3. 分别渲染两张极限帧的底片
    img_zero = render_single_frame(idx_zero, val_zero)
    img_max = render_single_frame(idx_max, val_max)

    # 🌟 4. 使用 matplotlib 上下分屏对比展示
    fig, axes = plt.subplots(2, 1, figsize=(18, 12))

    axes[0].imshow(img_zero)
    axes[0].set_title(f"Gripper State ~ 0 (Frame {idx_zero} | State {val_zero:.4f})", fontsize=16, fontweight='bold')
    axes[0].axis('off')

    axes[1].imshow(img_max)
    axes[1].set_title(f"Gripper Maximum Value (Frame {idx_max} | State {val_max:.4f})", fontsize=16, fontweight='bold')
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()

# ================= 启动验证 =================
pb_renderer_ultimate = PyBulletRenderer_Robotiq()

inspect_gripper_extremes(
    scene_constants=scene_constants,
    scene_state=stage3_scene_state,
    pb_renderer=pb_renderer_ultimate,
    tgt_width=1800
)

inspect_gripper_extremes(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
    pb_renderer=pb_renderer_ultimate,
    tgt_width=1800
)

In [ ]:
# @title 2D 机械臂分割可视化 (修复夹爪动态开合版)

def render_segmentation_video(
    scene_constants,
    scene_state,
    pb_renderer,
    tgt_width=1200
):
    """
    将 PyBullet 物理引擎中机械臂的三维姿态，重投影并叠加到 2D 真实视频上
    """
    # 🌟 1. 自动提取全局状态
    camera_ids = list(scene_constants['camera'].keys())
    wrist_serial = scene_constants['meta']['wrist_serial']

    # 动态获取总帧数
    n_frames = len(scene_constants['camera'][camera_ids[0]]['video_rgb'])
    video_frames = []

    # 🌟 2. 影视级逐帧渲染循环
    for frame_idx in tqdm(range(n_frames), desc=f"🎥 渲染分割视频"):

        # 🌟 核心修复：同时提取当前帧的机械臂关节角度和夹爪开合度
        current_joints = scene_constants['robot']['joint_positions'][frame_idx]
        current_gripper = scene_constants['robot']['gripper_positions'][frame_idx]

        # 强制物理引擎中的机械臂摆出当前真实关节姿态 (包含夹爪！)
        pb_renderer.update_robot_pose(current_joints, gripper_state=current_gripper)

        frame_views = []

        for cam_id in camera_ids:
            cam_data = scene_constants['camera'][cam_id]
            cam_state = scene_state[cam_id]

            img_rgb = cam_data['video_rgb'][frame_idx].copy()
            h_img, w_img = img_rgb.shape[:2]

            # 🌟 3. 核心魔法：利用最新位姿在物理引擎中拍下绝对精确的遮罩
            robot_mask = pb_renderer.render_mask(
                extrinsics=cam_state['extrinsics'][frame_idx],
                intrinsics=cam_data['K_mat'],
                width=w_img,
                height=h_img
            ) > 0

            # 🌟 4. 赛博朋克风混色：为机械臂穿上蓝色半透明紧身衣
            overlay = img_rgb.copy()
            overlay[robot_mask] = [50, 150, 255]
            blended_img = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

            # 🌟 5. 极简文本与阴影绘制 (防止高光背景看不清文字)
            is_wrist = (cam_id == wrist_serial)
            label_color = (0, 255, 255) if is_wrist else (0, 255, 0)

            cv2.putText(blended_img, f"Cam [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 0), 4)
            cv2.putText(blended_img, f"Cam [{cam_id}]", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255, 255, 255), 2)

            frame_views.append(blended_img)

        # 🌟 6. 横向拼接当前帧的所有机位，并进行等比例缩放
        row_concat = np.concatenate(frame_views, axis=1)
        tgt_height = int(row_concat.shape[0] * (tgt_width / row_concat.shape[1]))
        video_frames.append(cv2.resize(row_concat, (tgt_width, tgt_height)))

    return video_frames

# ================= 启动渲染 =================
pb_renderer_ultimate = PyBulletRenderer_Robotiq()

seg_video_frames = render_segmentation_video(
    scene_constants=scene_constants,
    scene_state=stage3_scene_state,
    pb_renderer=pb_renderer_ultimate,
    tgt_width=1200
)

media.show_video(seg_video_frames, fps=15, codec='gif')

seg_video_frames = render_segmentation_video(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
    pb_renderer=pb_renderer_ultimate,
    tgt_width=1200
)

media.show_video(seg_video_frames, fps=15, codec='gif')

In [ ]:
# @title 2D 联合交叉绘制相机坐标轴验证

def render_cross_camera_axes(scene_constants, scene_state, axis_len=0.15, tgt_w=1200):
    # 🌟 1. 语义复苏：直接从全新的 camera 层级提取相机列表，干掉丑陋的 if 过滤
    cams = list(scene_constants['camera'].keys())
    n_frames = len(scene_state[cams[0]]['extrinsics'])

    # 巧妙构造局部齐次坐标轴矩阵 (4x4) -> [原点, X, Y, Z]
    axes_3d = np.array([[0, 0, 0, 1], [axis_len, 0, 0, 1],
                        [0, axis_len, 0, 1], [0, 0, axis_len, 1]]).T

    video_frames = []

    # 🌟 2. 纯粹的渲染循环：保留清晰的帧索引与相机迭代，加入进度条避免无聊等待
    for frame_idx in tqdm(range(n_frames), desc="🎥 渲染交叉坐标轴"):
        camera_views = []

        for obs_cam in cams:
            cam_data = scene_constants['camera'][obs_cam]
            img_rgb = cam_data['video_rgb'][frame_idx].copy()
            h_img, w_img = img_rgb.shape[:2]
            K_mat = cam_data['K_mat']

            # 观察视角的逆矩阵 (用于将目标相机坐标变换到观察相机坐标系)
            obs_pose_inv = np.linalg.inv(scene_state[obs_cam]['extrinsics'][frame_idx])

            for tgt_cam in cams:
                if obs_cam == tgt_cam:
                    continue

                # 🌟 核心魔法：一行完成 3D 相对变换与相机投影
                tgt_pose = scene_state[tgt_cam]['extrinsics'][frame_idx]
                pts_cam = (obs_pose_inv @ tgt_pose @ axes_3d)[:3, :]

                if pts_cam[2, 0] < 0:
                    continue  # 剔除在相机背后的点

                # 🌟 极简解包：利用 map 和 转置(T) 直接获取 OpenCV 需要的 tuple
                uv = K_mat @ pts_cam
                org, px, py, pz = map(tuple, (uv[:2] / uv[2]).astype(int).T)

                # 若原点在画面内，则绘制红绿蓝坐标轴
                if 0 <= org[0] < w_img and 0 <= org[1] < h_img:
                    cv2.line(img_rgb, org, px, (255, 0, 0), 3)
                    cv2.line(img_rgb, org, py, (0, 255, 0), 3)
                    cv2.line(img_rgb, org, pz, (0, 0, 255), 3)
                    cv2.circle(img_rgb, org, 5, (0, 0, 0), -1)
                    cv2.circle(img_rgb, org, 2, (255, 255, 255), -1)

                    cv2.putText(img_rgb, f"Cam {tgt_cam}", (org[0]+8, org[1]-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 3)
                    cv2.putText(img_rgb, f"Cam {tgt_cam}", (org[0]+8, org[1]-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

            # 水印绘制
            cv2.putText(img_rgb, f"View: {obs_cam}", (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 3)
            cv2.putText(img_rgb, f"View: {obs_cam}", (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 255), 2)
            camera_views.append(img_rgb)

        # 🌟 3. 极简拼接与等比例缩放：将变量名换为具备画面感的表述
        row_concat = np.concatenate(camera_views, axis=1)
        tgt_h = int(row_concat.shape[0] * tgt_w / row_concat.shape[1])
        video_frames.append(cv2.resize(row_concat, (tgt_w, tgt_h)))

    return video_frames

all_viz_frames = render_cross_camera_axes(
    scene_constants=scene_constants,
    scene_state=stage3_scene_state,
)

media.show_video(all_viz_frames, fps=15, codec='gif')

all_viz_frames = render_cross_camera_axes(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
)

media.show_video(all_viz_frames, fps=15, codec='gif')

In [ ]:
# @title 3D 单帧融合点云与交互式可视化

render_fused_point_cloud(
    scene_constants=scene_constants,
    scene_state=stage3_scene_state,
)

render_fused_point_cloud(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
)

In [ ]:
# @title 4D 全域点云环绕视频

def get_look_at_matrix(eye, target, up=(0, 0, 1)):
    """3D 运镜辅助：一行流生成 OpenGL 风格相机朝向矩阵"""
    z_axis = np.array(eye, dtype=float) - target
    z_axis /= np.linalg.norm(z_axis) + 1e-6

    x_axis = np.cross(up, z_axis)
    x_axis /= np.linalg.norm(x_axis) + 1e-6

    y_axis = np.cross(z_axis, x_axis)

    view_matrix = np.eye(4)
    view_matrix[:3, :4] = np.column_stack((x_axis, y_axis, z_axis, eye))
    return view_matrix

def render_cinematic_4d_orbit(scene_constants, scene_state, max_render_points=400000, width=640,
                              height=360, orbit_center=(0.4, 0.0, 0.0), orbit_radius=1.2,
                              camera_height=0.5, angle_start=np.pi/2):

    # 🌟 1. 拥抱新架构：直接提取 camera 层级的 keys，彻底干掉丑陋的 if 过滤
    camera_ids = sorted(scene_constants['camera'].keys())
    n_frames = len(scene_state[camera_ids[0]]['extrinsics'])

    # 🌟 2. 场景与渲染器初始化
    scene = pyrender.Scene(bg_color=[0.0, 0.0, 0.0, 1.0])
    cam_node = scene.add(pyrender.PerspectiveCamera(yfov=np.pi/3.0, aspectRatio=width/height), pose=np.eye(4))
    light_node = scene.add(pyrender.DirectionalLight(color=[1.0, 1.0, 1.0], intensity=4.0), pose=np.eye(4))
    renderer = pyrender.OffscreenRenderer(width, height)

    video_frames = []

    # 🌟 3. 极速 4D 渲染循环：加入语义化变量
    for frame_idx in tqdm(range(n_frames), desc=f"🎥 渲染 4D 运镜"):

        # --- A. 提取并融合多视角点云 ---
        points, colors = [], []
        for cam_id in camera_ids:
            cam_data = scene_constants['camera'][cam_id]
            cam_state = scene_state[cam_id]

            # 提取指定帧的 3D 空间点与色彩
            points_3d, colors_rgb = unproject_to_3d(
                cam_data['raw_depth'][frame_idx],
                cam_data['video_rgb'][frame_idx],
                cam_data['K_mat'],
                T_cam2world=cam_state['extrinsics'][frame_idx]
            )
            points.append(points_3d)
            colors.append(colors_rgb)

        points = np.vstack(points)
        colors = np.vstack(colors)

        # --- B. 显存保护机制 (降维打击，只需两行) ---
        sample_idx = np.random.permutation(len(points))[:max_render_points]
        points, colors = points[sample_idx], colors[sample_idx]

        # --- C. 计算平滑环绕运镜位姿 ---
        angle = angle_start + (frame_idx * np.pi / n_frames)
        eye_pos = [orbit_center[0] + orbit_radius * np.cos(angle),
                   orbit_center[1] + orbit_radius * np.sin(angle),
                   camera_height]

        viz_pose = get_look_at_matrix(eye_pos, orbit_center)
        scene.set_pose(cam_node, pose=viz_pose)
        scene.set_pose(light_node, pose=viz_pose)

        # --- D. 压入渲染、拔出销毁、盖水印一气呵成 ---
        mesh_node = scene.add(pyrender.Mesh.from_points(points, colors=colors))
        color_img, _ = renderer.render(scene, flags=pyrender.RenderFlags.RGBA)
        scene.remove_node(mesh_node)

        # 🌟 仅拷贝 RGB 通道打断只读锁定，跳过 Alpha 通道的冗余拷贝
        img_rgb = color_img[:, :, :3].copy()
        cv2.putText(img_rgb, f"Frame: {frame_idx:03d}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)

        video_frames.append(img_rgb)

    renderer.delete()
    return video_frames

# ================= 主流程 =================
output_frames = render_cinematic_4d_orbit(
    scene_constants=scene_constants,
    scene_state=stage3_scene_state,
)

media.show_video(output_frames, fps=15, codec='gif')

output_frames = render_cinematic_4d_orbit(
    scene_constants=scene_constants,
    scene_state=ultimate_scene_state,
)

media.show_video(output_frames, fps=15, codec='gif')

### 相机外參标定 PointWorld 提速版 (不work)

In [ ]:

import yourdfpy
import trimesh
import torch
import numpy as np

class TensorRobotRenderer:
    """
    极速版网格采样渲染器 (yourdfpy 终极安全防弹版)
    利用 nodes_geometry 完美解析 Scene Graph，彻底告别所有底层属性断层！
    """
    def __init__(self, urdf_path, device="cuda", total_samples=15000):
        self.device = device
        self.dtype = torch.float32
        self.total_samples = total_samples

        print(f"⚡ [Hybrid Engine] 正在加载 yourdfpy 现代化张量模型...")
        self.robot = yourdfpy.URDF.load(urdf_path)

        self.mesh_points = {}
        self.fk_cache = {}
        self.world_points_cache = {}

    def _sample_mesh(self, fk_poses):
        """根据网格面积比例，在机器人表面预撒点"""
        mesh_names, mesh_objs, mesh_areas = [], [], []

        for geom_name in fk_poses.keys():
            mesh = self.robot.scene.geometry.get(geom_name)
            if mesh is None or mesh.area <= 0:
                continue

            area = mesh.area
            # 压低手部相机及碰撞体外壳的采样权重，防止拉偏外参优化
            if 'hand_camera' in geom_name.lower() or 'camera' in geom_name.lower():
                area *= 0.0001

            mesh_names.append(geom_name)
            mesh_objs.append(mesh)
            mesh_areas.append(area)

        total_area = sum(mesh_areas)
        for name, mesh, area in zip(mesh_names, mesh_objs, mesh_areas):
            count = max(50, int(self.total_samples * area / total_area))
            pts, _ = trimesh.sample.sample_surface(mesh, count)
            self.mesh_points[name] = torch.tensor(pts, dtype=self.dtype, device=self.device)

    def get_world_points(self, joint_positions, gripper_state):
        """主入口：输入关节状态，直接输出全机身 3D 真值点云 (N, 3)"""
        cache_key = tuple(np.round(joint_positions, 4).tolist() + [round(float(gripper_state), 4)])
        if cache_key in self.world_points_cache:
            return self.world_points_cache[cache_key]

        # 1. 组装当前关节配置
        cfg = {'finger_joint': float(gripper_state)}
        for i in range(7):
            cfg[f'panda_joint{i+1}'] = float(joint_positions[i])

        # 2. 获取正向运动学位姿
        if cache_key in self.fk_cache:
            fk_poses = self.fk_cache[cache_key]
        else:
            self.robot.update_cfg(cfg)
            fk_poses = {}
            # 🌟 终极排雷：使用 nodes_geometry 安全遍历所有带有 3D 几何体的节点
            # 这能完美避开 'URDF' object has no attribute 或是 dict_keys 的报错
            for node_name, geom_name in self.robot.scene.graph.nodes_geometry.items():
                transform, _ = self.robot.scene.graph.get(node_name)
                fk_poses[geom_name] = transform
            self.fk_cache[cache_key] = fk_poses

        # 如果尚未采样，则执行一次表面散点生成
        if not self.mesh_points:
            self._sample_mesh(fk_poses)

        # 3. 纯张量齐次变换推演
        all_pts = []
        for geom_name, pose_mat in fk_poses.items():
            if geom_name not in self.mesh_points:
                continue
            pose = torch.tensor(pose_mat, dtype=self.dtype, device=self.device)
            local_pts = self.mesh_points[geom_name]
            ones = torch.ones((local_pts.shape[0], 1), device=self.device, dtype=self.dtype)
            pts_h = torch.cat([local_pts, ones], dim=1)
            # P_world = T @ P_local
            world_pts = torch.mm(pts_h, pose.T)[:, :3]
            all_pts.append(world_pts)

        # 拼接并降采样对齐目标张量尺寸
        out_pts = torch.cat(all_pts, dim=0)
        idx = torch.randperm(out_pts.shape[0], device=self.device)[:2000]
        out_pts = out_pts[idx]

        self.world_points_cache[cache_key] = out_pts
        return out_pts

In [ ]:
def compute_robot_loss_batched_hybrid(batch_X, T_opt, K, batch_obs):
    """适配张量引擎的 Batched 损失计算 (引入动态截断抵抗穿模)"""
    B, _, h_img, w_img = batch_obs.shape
    P_c = (batch_X - T_opt[:3, 3]) @ T_opt[:3, :3]
    Z_pred = P_c[..., 2]

    u = K[0, 0] * P_c[..., 0] / Z_pred + K[0, 2]
    v = K[1, 1] * P_c[..., 1] / Z_pred + K[1, 2]
    grid = torch.stack([(u / (w_img - 1)) * 2 - 1, (v / (h_img - 1)) * 2 - 1], dim=-1).unsqueeze(1)

    Z_obs_raw = F.grid_sample(batch_obs, grid, mode='bilinear', padding_mode='border', align_corners=True).squeeze(1).squeeze(1)

    # 基础边界掩码
    valid_mask = (Z_pred > 0.) & (Z_pred < 1.5) & (Z_obs_raw > 0.) & (Z_obs_raw < 1.5) & \
                 (u >= 0) & (u < w_img - 1) & (v >= 0) & (v < h_img - 1)

    diff = torch.abs(Z_obs_raw - Z_pred)
    # 🌟 核心排爆：通过 15cm 物理公差，强制过滤掉机械臂背面的点和背景穿模点！
    valid_mask = valid_mask & (diff < 0.15)

    return torch.nan_to_num(diff[valid_mask].mean(), nan=0.0)

def run_stage2_robot_alignment_hybrid(scene_constants, tensor_renderer, init_scene_state=None, vggt_scene_state=None):
    OUTER_LOOPS = 5
    INNER_LOOPS = 100
    print(f"\n🦾 启动第一阶段优化 (🚀 Hybrid 张量狂飙引擎接入!)...")

    ext_cams = [c for c in scene_constants['camera'].keys() if c != scene_constants['meta']['wrist_serial']]
    n_frames = len(scene_constants['camera'][ext_cams[0]]['video_rgb'])

    def make_T(delta):
        T = torch.eye(4, device=device)
        T[:3, :3], T[:3, 3] = axis_angle_to_matrix(delta[:3]), delta[3:]
        return T

    has_init = init_scene_state is not None
    pybullet_scene_state = copy.deepcopy(init_scene_state if has_init else vggt_scene_state)

    def optimize_camera_with_base(cam_id, base_extrinsic):
        T_init_t = torch.tensor(base_extrinsic, dtype=torch.float32, device=device)
        K_t = torch.tensor(scene_constants['camera'][cam_id]['K_mat'], dtype=torch.float32, device=device)
        d_ext = torch.zeros(6, requires_grad=True, device=device)
        optimizer = optim.Adam([d_ext], lr=0.002) # 张量引擎极度稳定，可以稍微拉高学习率
        final_loss = float('inf')

        for outer_step in range(OUTER_LOOPS):
            cache_X, cache_obs = [], []
            # 🌟 核心提速：使用张量引擎直接生成绝对 GT 物理点！拔掉 pybullet！
            for t in range(n_frames):
                joints = scene_constants['robot']['joint_positions'][t]
                gripper = scene_constants['robot']['gripper_positions'][t]

                # 直接抽取绝对 3D 点云
                cad_pts_3d = tensor_renderer.get_world_points(joints, gripper)
                d_obs = scene_constants['camera'][cam_id]['raw_depth'][t].astype(np.float32)

                cache_X.append(cad_pts_3d)
                cache_obs.append(torch.tensor(d_obs, dtype=torch.float32, device=device)[None, ...])

            batch_X = torch.stack(cache_X)
            batch_obs = torch.stack(cache_obs)

            for inner_step in range(INNER_LOOPS):
                optimizer.zero_grad()
                loss_rob = compute_robot_loss_batched_hybrid(batch_X, T_init_t @ make_T(d_ext), K_t, batch_obs)
                loss_rob.backward()
                optimizer.step()
                final_loss = loss_rob.item()

                if inner_step % 50 == 0 or inner_step == INNER_LOOPS - 1:
                    print(f"      Outer {outer_step+1}/{OUTER_LOOPS} | Inner {inner_step:03d} | Tensor Loss: {final_loss:.4f}")

        with torch.no_grad():
            T_final_np = (T_init_t @ make_T(d_ext)).cpu().numpy()
            shift_mm = np.linalg.norm(d_ext[3:].detach().cpu().numpy()) * 1000
        return T_final_np, final_loss, shift_mm

    # --- 后续擂台逻辑保持不变 ---
    for cam in ext_cams:
        print(f"\n  📷 正在独立优化外部相机: [{cam}] ...")
        base_candidates = [("VGGT", vggt_scene_state[cam]['base_extrinsic'])]
        if has_init: base_candidates.append(("Dataset Init", init_scene_state[cam]['base_extrinsic']))

        best_loss, best_T, best_source_name, best_shift = float('inf'), None, None, 0.0

        for source_name, base_ext in base_candidates:
            print(f"    --> 🔄 尝试基于 [{source_name}] 基底进行极速解算:")
            T_res, loss_res, shift_res = optimize_camera_with_base(cam, base_ext)
            if loss_res < best_loss:
                best_loss, best_T, best_source_name, best_shift = loss_res, T_res, source_name, shift_res

        print(f"  ✅ [{cam}] 刚体对齐完毕！采纳最优基底: {best_source_name}, 最终 Loss: {best_loss:.4f} (相对修正: {best_shift:.2f}mm)")
        pybullet_scene_state[cam]['base_extrinsic'] = best_T
        pybullet_scene_state[cam]['extrinsics'] = np.tile(best_T, (n_frames, 1, 1))

    return pybullet_scene_state

# ================= 启动调用 =================
# 1. 实例化极速引擎
urdf_path = "PointWorld/assets/franka_description/franka_panda_robotiq_2f85_og.urdf"
tensor_renderer_ultimate = TensorRobotRenderer(urdf_path, device=device)

# 2. 传入混合引擎跑 Stage 2
pybullet_scene_state = run_stage2_robot_alignment_hybrid(
    scene_constants=scene_constants,
    tensor_renderer=tensor_renderer_ultimate,  # 注入张量引擎
    init_scene_state=init_scene_state if all_extrinsics_exist else None,
    vggt_scene_state=vggt_scene_state
)

In [ ]:
def compute_wrist_loss_batched_hybrid(batch_P_ee, T_cam_ee_opt, K, batch_obs):
    """🦇 Wrist 专属混合引擎算子：引入 5cm 动态公差抵抗指尖穿模"""
    B, _, h_img, w_img = batch_obs.shape
    T_ee_cam = torch.linalg.inv(T_cam_ee_opt)

    # 锚定在末端(EE)的点拨回当前相机坐标系
    P_c = batch_P_ee @ T_ee_cam[:3, :3].T + T_ee_cam[:3, 3]
    Z_pred = P_c[..., 2]

    u = K[0, 0] * P_c[..., 0] / Z_pred + K[0, 2]
    v = K[1, 1] * P_c[..., 1] / Z_pred + K[1, 2]
    grid = torch.stack([(u / (w_img - 1)) * 2 - 1, (v / (h_img - 1)) * 2 - 1], dim=-1).unsqueeze(1)

    Z_obs_raw = F.grid_sample(batch_obs, grid, mode='bilinear', padding_mode='border', align_corners=True).squeeze(1).squeeze(1)

    # 基础掩码：限制在 1 米内
    valid_mask = (Z_pred > 0.) & (Z_pred < 1.0) & (Z_obs_raw > 0.) & (Z_obs_raw < 1.0) & \
                 (u >= 0) & (u < w_img - 1) & (v >= 0) & (v < h_img - 1)

    diff = torch.abs(Z_obs_raw - Z_pred)
    # 🌟 终极排爆：夹爪体积很小，5cm 以上的误差大概率是投射到了手指背面或者是桌面上，强行截断！
    valid_mask = valid_mask & (diff < 0.05)

    return torch.nan_to_num(diff[valid_mask].mean(), nan=0.0)

def run_stage2_wrist_alignment_hybrid(scene_constants, init_scene_state, tensor_renderer):
    OUTER_LOOPS = 5
    INNER_LOOPS = 100
    MAX_ROBOT_PTS = 2000
    print(f"\n🦾 启动腕部实验：纯夹爪物理锚定 (🚀 Hybrid 张量狂飙引擎全覆盖!)...")

    wrist_cam = scene_constants['meta']['wrist_serial']
    n_frames = len(scene_constants['camera'][wrist_cam]['video_rgb'])
    pybullet_scene_state = copy.deepcopy(init_scene_state)
    h_img, w_img = scene_constants['camera'][wrist_cam]['raw_depth'][0].shape

    T_cam_ee_init_np = init_scene_state[wrist_cam]['base_extrinsic']
    T_cam_ee_init_t = torch.tensor(T_cam_ee_init_np, dtype=torch.float32, device=device)
    K_np = scene_constants['camera'][wrist_cam]['K_mat']
    K_t = torch.tensor(K_np, dtype=torch.float32, device=device)
    T_ee_base_all = scene_constants['robot']['T_ee_base_all']

    d_ext = torch.zeros(6, requires_grad=True, device=device)
    optimizer = optim.Adam([d_ext], lr=0.001)

    def make_T(delta):
        T = torch.eye(4, device=device)
        T[:3, :3], T[:3, 3] = axis_angle_to_matrix(delta[:3]), delta[3:]
        return T

    for outer_step in range(OUTER_LOOPS):
        with torch.no_grad():
            T_cam_ee_np = (T_cam_ee_init_t @ make_T(d_ext)).cpu().numpy()

        cache_P_ee, cache_obs = [], []
        for t in range(n_frames):
            joints = scene_constants['robot']['joint_positions'][t]
            gripper = scene_constants['robot']['gripper_positions'][t]

            # 1. 抽取全机身绝对 3D 点云
            cad_pts_world = tensor_renderer.get_world_points(joints, gripper)

            # 2. 转换到当前的【腕部相机坐标系】
            T_cam_world_np = T_ee_base_all[t] @ T_cam_ee_np
            T_cam_world_t = torch.tensor(T_cam_world_np, dtype=torch.float32, device=device)

            cad_pts_world_h = torch.cat([cad_pts_world, torch.ones((cad_pts_world.shape[0], 1), device=device)], dim=1)
            pts_cam = (T_cam_world_t @ cad_pts_world_h.T).T[:, :3]

            # 🌟 3. 视场 (FOV) 截断大法：留在视野内且 Z < 0.5 米的，必定是夹爪本体！
            Z = pts_cam[:, 2]
            u = K_t[0, 0] * pts_cam[:, 0] / Z + K_t[0, 2]
            v = K_t[1, 1] * pts_cam[:, 1] / Z + K_t[1, 2]

            valid_fov = (Z > 0.02) & (Z < 0.5) & (u >= 0) & (u < w_img) & (v >= 0) & (v < h_img)
            pts_cam_valid = pts_cam[valid_fov]

            if pts_cam_valid.shape[0] < 50:
                continue

            # 4. 将视野内合法的夹爪点转移到【末端执行器 EE】坐标系中固化 (这是为了给 Inner Loop 优化器用)
            T_cam_ee_t = torch.tensor(T_cam_ee_np, dtype=torch.float32, device=device)
            T_ee_cam_t = torch.linalg.inv(T_cam_ee_t)

            pts_cam_valid_h = torch.cat([pts_cam_valid, torch.ones((pts_cam_valid.shape[0], 1), device=device)], dim=1)
            P_ee_r = (T_ee_cam_t @ pts_cam_valid_h.T).T[:, :3]

            # 5. 随机降采样 / Pad 对齐 Batch Shape
            if P_ee_r.shape[0] >= MAX_ROBOT_PTS:
                idx = torch.randperm(P_ee_r.shape[0], device=device)[:MAX_ROBOT_PTS]
            else:
                idx = torch.randint(0, P_ee_r.shape[0], (MAX_ROBOT_PTS,), device=device)
            P_ee_r = P_ee_r[idx]

            d_obs = scene_constants['camera'][wrist_cam]['raw_depth'][t].astype(np.float32)
            cache_P_ee.append(P_ee_r)
            cache_obs.append(torch.tensor(d_obs, dtype=torch.float32, device=device)[None, ...])

        if not cache_P_ee:
            print(f"    ⚠️ 糟糕，当前帧找不到能够投射到画面的夹爪点云！")
            break

        batch_P_ee = torch.stack(cache_P_ee)
        batch_obs = torch.stack(cache_obs)

        # Inner Loop 张量梯度狂飙
        for inner_step in range(INNER_LOOPS):
            optimizer.zero_grad()
            T_cam_ee_opt = T_cam_ee_init_t @ make_T(d_ext)
            loss_rob = compute_wrist_loss_batched_hybrid(batch_P_ee, T_cam_ee_opt, K_t, batch_obs)
            loss_rob.backward()
            optimizer.step()

            if inner_step % 50 == 0 or inner_step == INNER_LOOPS - 1:
                with torch.no_grad():
                    rot_deg = torch.norm(d_ext[:3]).item() * (180.0 / np.pi)
                    shift_mm = torch.norm(d_ext[3:]).item() * 1000.0
                print(f"    Outer {outer_step+1}/{OUTER_LOOPS} | Inner {inner_step:03d} | Wrist Tensor Loss: {loss_rob.item():.4f} | Shift: {shift_mm:.2f}mm | Rot: {rot_deg:.2f}°")

    with torch.no_grad():
        T_cam_ee_final = (T_cam_ee_init_t @ make_T(d_ext)).cpu().numpy()
        shift_mm = torch.norm(d_ext[3:]).item() * 1000.0
        rot_deg = torch.norm(d_ext[:3]).item() * (180.0 / np.pi)
        print(f"  ✅ [Wrist Cam: {wrist_cam}] 极速张量手眼标定收敛！修正量 -> 平移: {shift_mm:.2f}mm, 旋转: {rot_deg:.2f}°")

        pybullet_scene_state[wrist_cam]['base_extrinsic'] = T_cam_ee_final
        pybullet_scene_state[wrist_cam]['extrinsics'] = T_ee_base_all @ T_cam_ee_final

    return pybullet_scene_state

# ================= 启动调用 =================
# 接手第一步产生的 pybullet_scene_state，覆盖升级腕部相机外参
wrist_optimized_state = run_stage2_wrist_alignment_hybrid(
    scene_constants=scene_constants,
    init_scene_state=pybullet_scene_state,
    tensor_renderer=tensor_renderer_ultimate # 复用同一台极速引擎
)

In [ ]:
import pytorch_kinematics as pk
import yourdfpy
import trimesh
import torch
import numpy as np

class PKBatchedRobotRenderer:
    """
    史诗级 GPU 批处理渲染器
    在显存中一键计算 100+ 帧的正向运动学，瞬间吐出 4D 点云。
    """
    def __init__(self, urdf_path, device="cuda", points_per_link=200):
        self.device = device
        self.dtype = torch.float32

        print(f"🚀 [PK Engine] 正在构建 GPU 运动学图谱...")
        with open(urdf_path, "r") as f:
            self.chain = pk.build_chain_from_urdf(f.read()).to(dtype=self.dtype, device=self.device)

        print(f"🚀 [PK Engine] 正在将局部 3D 几何推入显存...")
        self.robot = yourdfpy.URDF.load(urdf_path)
        self.link_points = {}

        for link in self.robot.links:
            all_pts = []
            for visual in link.visuals:
                mesh = visual.geometry.mesh
                if mesh is None: continue
                # 将几何体转换为 trimesh
                if hasattr(mesh, 'meshes'):
                    mesh = mesh.meshes[0]

                if mesh.area > 0:
                    count = points_per_link
                    if 'camera' in link.name.lower(): count = 10
                    pts, _ = trimesh.sample.sample_surface(mesh, count)

                    # 加上连杆内部的局部偏移
                    if visual.origin is not None:
                        pts_h = np.hstack([pts, np.ones((len(pts), 1))])
                        pts = (visual.origin @ pts_h.T).T[:, :3]
                    all_pts.append(pts)

            if all_pts:
                merged_pts = np.vstack(all_pts)
                pts_h = np.hstack([merged_pts, np.ones((len(merged_pts), 1))])
                # 保存为 (4, N) 的张量，方便后续 GPU 批量乘法
                self.link_points[link.name] = torch.tensor(pts_h, dtype=self.dtype, device=self.device).T

    def get_world_points_batched(self, batch_joints, batch_gripper):
        """
        全 GPU 批处理入口！
        batch_joints: (B, 7) 张量 (比如 184 帧的关节)
        batch_gripper: (B,) 张量
        返回: (B, 2000, 3) 张量 (184 帧的 3D 点云)
        """
        B = batch_joints.shape[0]

        # 组装 8 个关节维度的字典
        th_dict = {}
        for i in range(7):
            th_dict[f'panda_joint{i+1}'] = batch_joints[:, i]

        # Robotiq 夹爪全指节联动 (模仿 URDF 的 mimic 机制)
        finger_val = batch_gripper
        for joint in self.robot.joints:
            if 'finger' in joint.name or 'knuckle' in joint.name:
                th_dict[joint.name] = finger_val

        # 🌟 GPU 上的一次性正向运动学狂飙！
        ret = self.chain.forward_kinematics(th_dict)

        all_world_pts = []
        for link_name, pts_local_h in self.link_points.items():
            if link_name in ret:
                T_batched = ret[link_name].get_matrix() # 形如 (B, 4, 4)
                # 张量乘法：(B, 4, 4) @ (B, 4, N) -> (B, 4, N)
                pts_world_h = torch.bmm(T_batched, pts_local_h.unsqueeze(0).expand(B, -1, -1))
                # 转换回 (B, N, 3)
                all_world_pts.append(pts_world_h[:, :3, :].transpose(1, 2))

        # 拼接全机身所有点
        out_pts = torch.cat(all_world_pts, dim=1)

        # 随机抽取 2000 个点对齐 Shape
        idx = torch.randperm(out_pts.shape[1], device=self.device)[:2000]
        return out_pts[:, idx, :]

In [ ]:
# 将你的全时序 Numpy 数组转换为 PyTorch 张量
all_joints = torch.tensor(scene_constants['robot']['joint_positions'], dtype=torch.float32, device=device)
all_grippers = torch.tensor(scene_constants['robot']['gripper_positions'], dtype=torch.float32, device=device)

# 🚀 见证奇迹：一行代码拿到 184 帧的全机身 3D 点云！
# batch_X 的 shape 将直接是 (184, 2000, 3)
batch_X = pk_renderer.get_world_points_batched(all_joints, all_grippers)

# 原本的深度图 obs 也一样直接打包
batch_obs = torch.tensor(scene_constants['camera'][cam_id]['raw_depth'], dtype=torch.float32, device=device).unsqueeze(1)

# 然后直接丢进 Inner Loop 里去优化就行了！
for inner_step in range(INNER_LOOPS):
    optimizer.zero_grad()
    loss_rob = compute_robot_loss_batched_hybrid(batch_X, T_init_t @ make_T(d_ext), K_t, batch_obs)
    # ...